In [59]:
# @title ✅ SEL 1: Imports & Global Configuration (FIXED)
# ============================================================================
import os, sys, subprocess, math, random, time, threading, logging, textwrap
from collections import defaultdict, deque
from typing import List, Tuple, Dict, Optional, Set, Union, Callable, Any
import os, sys, subprocess, math, random, time, threading, logging
from collections import defaultdict, deque, OrderedDict
from typing import List, Tuple, Dict, Optional, Set, Union, Callable, Any

# Configure logging for production
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

def install(pkg: str) -> None:
    """Install package quietly if not already installed."""
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Auto-install and load spaCy
try:
    import spacy
except ImportError:
    install("spacy")
    import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")

# PyTorch & NumPy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

# Device detection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"🔧 Device: {device}")

# ============================================================================
# Global Parameters (Production-Configurable)
# ============================================================================
D = 64                          # Embedding dimension
LR = 0.1                        # LogicSlot learning rate
MOMENTUM = 0.9                  # Momentum for LogicSlot optimizer
LAMBDA = 0.01                   # L2 regularization
SIM_THRESHOLD = 0.9             # Similarity threshold for entity matching
MAX_SEQ_LEN = 32                # Max token sequence length
BATCH_SIZE = 16                 # Training batch size
EPOCHS = 50                     # Training epochs
NEURAL_LR = 0.001               # Neural network learning rate
ALIGNMENT_LAMBDA = 0.15         # Alignment loss weight
MAX_PLAN_LEN = 5                # Max reasoning plan length
RETRIEVAL_TOP_K = 3             # Top-K retrieval results
RETRIEVAL_CONF_THRESHOLD = 0.3  # Minimum retrieval confidence
ILP_CONFIDENCE = 0.8            # Base confidence for ILP inference
RULE_MIN_CONFIDENCE = 0.8       # Minimum confidence for rule mining
ANSWER_CONFIDENCE_THRESHOLD = 0.5  # Answer acceptance threshold
COMBINE_METHOD = "max"          # Confidence combination method
NEURAL_EXTRACTION_CONFIDENCE = 0.9  # Default extraction confidence
TRANSITIVE_RELATIONS = {"chase", "catch"}  # Relations supporting transitivity

GLOBAL_CONFIDENCE_THRESHOLD = ANSWER_CONFIDENCE_THRESHOLD

# ============================================================================
# Utility Functions
# ============================================================================
def normalize_entity_name(name: str) -> str:
    """Normalize entity name by lowercasing and removing articles."""
    name = name.strip().lower()
    for article in ("the ", "a ", "an "):
        if name.startswith(article):
            name = name[len(article):]
            break
    return name.strip()

def safe_read_lines(filepath: str, encoding: str = "utf-8") -> List[str]:
    """Safely read lines from file with error handling."""
    try:
        if not os.path.exists(filepath):
            logger.warning(f"File not found: {filepath}")
            return []
        with open(filepath, "r", encoding=encoding) as f:
            return [line.strip() for line in f if line.strip()]
    except PermissionError:
        logger.error(f"Permission denied: {filepath}")
        return []
    except UnicodeDecodeError:
        logger.warning(f"Encoding error, trying fallback: {filepath}")
        return safe_read_lines(filepath, encoding="latin-1")
    except Exception as e:
        logger.exception(f"Error reading {filepath}: {e}")
        return []

In [60]:
# @title ✅ SEL 2: Core Data Structures (EntityManager, RelationManager, DynamicTokenizer)
# ============================================================================

# ----------------------------------------------------------------------------
# EntityManager: Handles entity vectors with LSH-based approximate search
# ----------------------------------------------------------------------------
class EntityManager:
    def __init__(self, dim: int, initial_capacity: int = 1024, num_hashes: int = 8):
        self.dim = dim
        self.capacity = initial_capacity
        self.count = 0
        self.entities: List[str] = []
        self.entity2idx: Dict[str, int] = {}
        self.vectors = torch.empty((initial_capacity, dim), device=device)
        self.num_hashes = num_hashes
        self.planes = torch.randn((num_hashes, dim), device=device)
        self.buckets: Dict[Tuple[int, ...], List[int]] = {}
        self._on_new_entity_callbacks: List[Callable[[str, int], None]] = []

    def register_new_entity_callback(self, callback: Callable[[str, int], None]) -> None:
        self._on_new_entity_callbacks.append(callback)

    def _resize(self, new_capacity: int) -> None:
        new_vectors = torch.empty((new_capacity, self.dim), device=device)
        if self.count > 0:
            new_vectors[:self.count] = self.vectors[:self.count]
        self.vectors = new_vectors
        self.capacity = new_capacity

    def _compute_hash(self, vec: torch.Tensor) -> Tuple[int, ...]:
        with torch.no_grad():
            proj = torch.mv(self.planes, vec)
            return tuple((proj > 0).int().tolist())

    def get_or_create(self, name: str, context_vec: Optional[torch.Tensor] = None) -> Tuple[int, torch.Tensor]:
        name = normalize_entity_name(name)
        if name in self.entity2idx:
            return self.entity2idx[name], self.vectors[self.entity2idx[name]]

        if self.count >= self.capacity:
            self._resize(self.capacity * 2)

        idx = self.count
        self.entities.append(name)
        self.entity2idx[name] = idx

        if context_vec is not None:
            v = F.normalize(context_vec, dim=0)
        else:
            v = F.normalize(torch.randn(self.dim, device=device), dim=0)

        self.vectors[idx] = v
        h = self._compute_hash(v)
        self.buckets.setdefault(h, []).append(idx)
        self.count += 1

        for cb in self._on_new_entity_callbacks:
            cb(name, idx)

        return idx, v

    def get_vector(self, name: str) -> Optional[torch.Tensor]:
        name = normalize_entity_name(name)
        if name in self.entity2idx:
            return self.vectors[self.entity2idx[name]]
        return None

    def find_closest(self, vec: torch.Tensor, early_exit_thresh: float = 0.99) -> Tuple[int, str, float]:
        if self.count == 0:
            return -1, " ", -1.0

        h = self._compute_hash(vec)
        candidate_indices = set(self.buckets.get(h, []))

        # 🧠 Multi-Probe LSH Fallback: Probe neighbor buckets by flipping 1-2 bits
        if len(candidate_indices) < 5:
            h_list = list(h)
            n = len(h_list)

            # Probe 1-bit flips (direct LSH neighbors)
            for i in range(n):
                neighbor = list(h_list)
                neighbor[i] = 1 - neighbor[i]
                candidate_indices.update(self.buckets.get(tuple(neighbor), []))

            # Probe 2-bit flips if still sparse (wider neighborhood)
            if len(candidate_indices) < 20:
                for i in range(n):
                    for j in range(i + 1, n):
                        neighbor = list(h_list)
                        neighbor[i] = 1 - neighbor[i]
                        neighbor[j] = 1 - neighbor[j]
                        candidate_indices.update(self.buckets.get(tuple(neighbor), []))

        # Ultimate safety fallback: random sample only if LSH coverage is globally insufficient
        if len(candidate_indices) < 5 and len(candidate_indices) < self.count:
            max_candidates = min(200, self.count)
            remaining = [idx for idx in range(self.count) if idx not in candidate_indices]
            if remaining:
                candidate_indices.update(random.sample(remaining, min(len(remaining), max_candidates - len(candidate_indices))))

        candidates = list(candidate_indices)
        candidate_vectors = self.vectors[candidates]
        sims = torch.mv(candidate_vectors, vec)
        max_sim, local_idx = torch.max(sims, dim=0)
        best_idx = candidates[local_idx.item()]

        return best_idx, self.entities[best_idx], max_sim.item()


# ----------------------------------------------------------------------------
# RelationManager: Manages relation vocabulary
# ----------------------------------------------------------------------------
class RelationManager:
    def __init__(self):
        self.relations: List[str] = []
        self.rel2idx: Dict[str, int] = {}
        self.idx2rel: Dict[int, str] = {}
        self._on_new_relation_callbacks: List[Callable[[str, int], None]] = []

    def register_new_relation_callback(self, callback: Callable[[str, int], None]) -> None:
        self._on_new_relation_callbacks.append(callback)

    def get_or_create(self, rel: str) -> int:
        rel = rel.strip().lower()
        if rel not in self.rel2idx:
            idx = len(self.relations)
            self.relations.append(rel)
            self.rel2idx[rel] = idx
            self.idx2rel[idx] = rel
            for cb in self._on_new_relation_callbacks:
                cb(rel, idx)
            return idx
        return self.rel2idx[rel]

    def get_idx(self, rel: str) -> Optional[int]:
        rel = rel.strip().lower()
        return self.rel2idx.get(rel)

    def __len__(self) -> int:
        return len(self.relations)


# ----------------------------------------------------------------------------
# DynamicTokenizer: Vocabulary management with dynamic expansion
# ----------------------------------------------------------------------------
class DynamicTokenizer:
    def __init__(self, ent_mgr: EntityManager, rel_mgr: RelationManager):
        self.ent_mgr = ent_mgr
        self.rel_mgr = rel_mgr
        # ✅ FIX: Ganti string kosong/spasi dengan token standar NLP yang valid
        self.special_tokens = ["<pad>", "<unk>", ".", "?", "the", "a", "an", "in", "on", "to",
                                "was", "were", "is", "are", "what", "did", "where", "that",
                                "who", "after", "before", "with", "at", "from", "by"]
        self._build_vocab()
        ent_mgr.register_new_entity_callback(lambda name, idx: self._on_new_token(name))
        rel_mgr.register_new_relation_callback(lambda name, idx: self._on_new_token(name))

    def _on_new_token(self, token: str) -> None:
        if token not in self.word2idx:
            idx = len(self.word2idx)
            self.word2idx[token] = idx
            self.idx2word[idx] = token

    def _build_vocab(self) -> None:
        entities = self.ent_mgr.entities
        relations = self.rel_mgr.relations
        all_tokens = set(entities + relations + self.special_tokens)
        self.word2idx = {w: i for i, w in enumerate(sorted(all_tokens))}
        self.idx2word = {i: w for w, i in self.word2idx.items()}

    def tokenize(self, text: str) -> List[int]:
        tokens = text.lower().replace(".", " . ").replace("?", " ? ").split()
        # ✅ FIX: Fallback eksplisit ke token <unk> untuk OOV (Out-Of-Vocabulary)
        return [self.word2idx.get(t, self.word2idx["<unk>"]) for t in tokens]

    def __len__(self) -> int:
        return len(self.word2idx)

In [61]:
# @title ✅ SEL 3: Logic Components (LogicSlot, TransitiveReasoner, RuleMiner, KnowledgeModule, AnalyticMemory, MetaController, BeamPlanner)
# ============================================================================

# ----------------------------------------------------------------------------
# LogicSlot: Neural-symbolic relation slot with adaptive learning
# ----------------------------------------------------------------------------
class LogicSlot:
    def __init__(self, slot_id: int, dim: int, rel_idx: int,
                 lr: float = LR, reg: float = LAMBDA, momentum: float = MOMENTUM,
                 max_keys: int = 1000):
        self.slot_id = slot_id
        self.rel_idx = rel_idx
        self.dim = dim
        self.lr = lr
        self.reg = reg
        self.momentum = momentum
        self.max_keys = max_keys

        self.R = torch.randn((dim, dim), device=device) * 0.01
        self.velocity = torch.zeros_like(self.R)

        self.direct_facts: List[Tuple[str, str]] = []
        self.inferred_facts: Set[Tuple[str, str]] = set()
        self.direct_subj_to_obj: Dict[str, str] = {}
        self.direct_obj_to_subj: Dict[str, str] = {}
        self.inferred_subj_to_objs: Dict[str, List[Tuple[str, float]]] = {}
        self.inferred_confidences: Dict[Tuple[str, str], float] = {}

        # ✅ OPTIMIZED: Index-based storage replaces vector duplication
        self.subj_indices: List[int] = []
        self.obj_names: List[str] = []

        self.error_count = 0
        self.success_count = 0
        self.adaptation_factor = 1.0
        self.performance_window = deque(maxlen=50)

    def _predict(self, x: torch.Tensor) -> torch.Tensor:
        return x @ (self.R * self.adaptation_factor)

    def update_performance(self, success: bool, confidence: float = 1.0) -> None:
        self.performance_window.append(success)
        if len(self.performance_window) >= 10:
            recent_accuracy = sum(self.performance_window) / len(self.performance_window)
            if recent_accuracy < 0.7:
                self.adaptation_factor = 0.95
            elif recent_accuracy > 0.9:
                self.adaptation_factor = 1.02
            self.adaptation_factor = max(0.5, min(2.0, self.adaptation_factor))

    def can_add(self, subj: str, obj: str, ent_mgr: EntityManager) -> Tuple[bool, Optional[str]]:
        if subj in self.direct_subj_to_obj and self.direct_subj_to_obj[subj] != obj:
            return False, f"Konflik direct: '{subj}' -> '{self.direct_subj_to_obj[subj]}'"
        if len(self.direct_facts) == 0:
            return True, None
        x_vec = ent_mgr.get_vector(subj)
        if x_vec is None:
            return True, None
        pred = self._predict(x_vec)
        pred_norm = F.normalize(pred, dim=0)
        _, closest_name, sim = ent_mgr.find_closest(pred_norm)
        if sim > SIM_THRESHOLD and closest_name != obj:
            return False, f"Prediksi neural kuat ke '{closest_name}'"
        return True, None

    def add_fact(self, subj: str, obj: str, ent_mgr: EntityManager,
                 is_inferred: bool = False, confidence: float = 1.0) -> bool:
        subj = normalize_entity_name(subj)
        obj = normalize_entity_name(obj)
        x_vec = ent_mgr.get_vector(subj)
        y_vec = ent_mgr.get_vector(obj)
        if x_vec is None or y_vec is None:
            return False

        if is_inferred:
            if subj in self.direct_subj_to_obj:
                return False
            existing = self.inferred_subj_to_objs.get(subj, [])
            for (ex_obj, ex_conf) in existing:
                if ex_obj == obj:
                    if confidence <= ex_conf:
                        return False
                    else:
                        self.inferred_facts.discard((subj, obj))
                        existing.remove((ex_obj, ex_conf))
                        break
            self.inferred_facts.add((subj, obj))
            self.inferred_subj_to_objs.setdefault(subj, []).append((obj, confidence))
            self.inferred_confidences[(subj, obj)] = confidence
        else:
            can, _ = self.can_add(subj, obj, ent_mgr)
            if not can:
                return False
            self.direct_facts.append((subj, obj))
            self.direct_subj_to_obj[subj] = obj
            self.direct_obj_to_subj[obj] = subj
            if subj in self.inferred_subj_to_objs:
                for (old_obj, _) in self.inferred_subj_to_objs[subj]:
                    self.inferred_facts.discard((subj, old_obj))
                    self.inferred_confidences.pop((subj, old_obj), None)
                del self.inferred_subj_to_objs[subj]

            with torch.no_grad():
                y_pred = self._predict(x_vec)
                grad = -2.0 * torch.outer(x_vec, y_vec - y_pred) + 2.0 * self.reg * self.R
                self.velocity = self.momentum * self.velocity + grad
                self.R -= self.lr * self.velocity

        # ✅ INDEX OPTIMIZATION: Store indices & names instead of cloning tensors
        subj_idx = ent_mgr.entity2idx.get(subj)
        if subj_idx is None:
            subj_idx, _ = ent_mgr.get_or_create(subj, context_vec=x_vec)
        self.subj_indices.append(subj_idx)
        self.obj_names.append(obj)

        if len(self.subj_indices) > self.max_keys:
            trim = max(1, self.max_keys // 10)
            self.subj_indices = self.subj_indices[trim:]
            self.obj_names = self.obj_names[trim:]
            self.direct_facts = self.direct_facts[trim:]
        return True

    def query(self, subj: str, ent_mgr: EntityManager) -> Optional[Tuple[str, float]]:
        subj = normalize_entity_name(subj)
        if subj in self.direct_subj_to_obj:
            self.update_performance(True, 1.0)
            return self.direct_subj_to_obj[subj], 1.0
        if subj in self.inferred_subj_to_objs:
            candidates = self.inferred_subj_to_objs[subj]
            best = max(candidates, key=lambda x: x[1])
            self.update_performance(True, best[1])
            return best[0], best[1]

        x_vec = ent_mgr.get_vector(subj)
        if x_vec is None or len(self.subj_indices) == 0:
            self.update_performance(False, 0.0)
            return None

        x_norm = F.normalize(x_vec, dim=0)
        # ✅ FAST RETRIEVAL: Direct tensor indexing from shared ent_mgr
        indices = torch.tensor(self.subj_indices, device=device)
        vecs = ent_mgr.vectors[indices]
        sims = torch.mv(vecs, x_norm)
        best_idx = torch.argmax(sims).item()
        sim_val = sims[best_idx].item()

        if sim_val > RETRIEVAL_CONF_THRESHOLD:
            result = self.obj_names[best_idx], sim_val
            self.update_performance(True, sim_val)
            return result
        else:
            self.update_performance(False, sim_val)
            return None

    def inverse_query(self, obj: str, ent_mgr: EntityManager) -> Optional[Tuple[str, float]]:
        obj = normalize_entity_name(obj)
        if obj in self.direct_obj_to_subj:
            return self.direct_obj_to_subj[obj], 1.0
        best_conf = -1.0
        best_subj = None
        for s, o in self.inferred_facts:
            if o == obj:
                conf = self.inferred_confidences.get((s, o), ILP_CONFIDENCE)
                if conf > best_conf:
                    best_conf = conf
                    best_subj = s
        return (best_subj, best_conf) if best_subj else None

    def get_all_facts(self) -> List[Tuple[str, str]]:
        return self.direct_facts.copy()

    def get_fact_confidence(self, subj: str, obj: str) -> float:
        subj = normalize_entity_name(subj)
        obj = normalize_entity_name(obj)
        if subj in self.direct_subj_to_obj and self.direct_subj_to_obj[subj] == obj:
            return 1.0
        return self.inferred_confidences.get((subj, obj), 0.0)

    # 🆕 BATCH QUERY SYSTEM (Audit-Ready Integration)
    def batch_query(self, subj_list: List[str], ent_mgr: EntityManager) -> List[Optional[Tuple[str, float]]]:
        B = len(subj_list)
        results_map = {i: None for i in range(B)}

        valid_indices = []
        valid_subjects = []
        for idx, s in enumerate(subj_list):
            norm_s = normalize_entity_name(s)
            if norm_s in ent_mgr.entity2idx:
                valid_indices.append(ent_mgr.entity2idx[norm_s])
                valid_subjects.append(norm_s)

        if not valid_indices:
            return [results_map[i] for i in range(B)]

        q_vecs = ent_mgr.vectors[torch.tensor(valid_indices, device=device)]
        q_norm = F.normalize(q_vecs, dim=1)

        obj_names = self.obj_names
        if not obj_names:
            return [results_map[i] for i in range(B)]

        t_indices = [ent_mgr.entity2idx[o] for o in obj_names if o in ent_mgr.entity2idx]
        if not t_indices:
            return [results_map[i] for i in range(B)]

        t_vecs = ent_mgr.vectors[torch.tensor(t_indices, device=device)]
        t_norm = F.normalize(t_vecs, dim=1)

        scores = q_norm @ t_norm.T
        best_idx_in_t = torch.argmax(scores, dim=1)
        best_scores = torch.gather(scores, 1, best_idx_in_t.unsqueeze(1)).squeeze(1)

        matched_targets = [obj_names[t_indices.index(idx.item())] for idx in best_idx_in_t]

        current_b = 0
        for idx in range(B):
            if idx in valid_indices:
                sim_val = best_scores[current_b].item()
                if sim_val >= RETRIEVAL_CONF_THRESHOLD:
                    results_map[idx] = (matched_targets[current_b], sim_val)
                current_b += 1

        return [results_map[i] for i in range(B)]

    def batch_inverse_query(self, obj_list: List[str], ent_mgr: EntityManager) -> List[Optional[Tuple[str, float]]]:
        results = []
        obj_set = set(normalize_entity_name(o) for o in obj_list)

        inferred_cache = {}
        for subj, conf in self.inferred_confidences.items():
            _, obj = subj
            obj_n = normalize_entity_name(obj)
            if obj_n in obj_set:
                inferred_cache[obj_n] = (normalize_entity_name(subj), conf)

        for o in obj_list:
            norm_o = normalize_entity_name(o)
            if norm_o in self.direct_obj_to_subj:
                results.append((self.direct_obj_to_subj[norm_o], 1.0))
            elif norm_o in inferred_cache:
                results.append(inferred_cache[norm_o])
            else:
                results.append(None)
        return results


# ----------------------------------------------------------------------------
# TransitiveReasoner: Applies transitive closure inference
# ----------------------------------------------------------------------------
class TransitiveReasoner:
    def __init__(self, confidence: float = ILP_CONFIDENCE):
        self.confidence = confidence

    def apply(self, slot: LogicSlot, ent_mgr: EntityManager, rel_name: str) -> int:
        if rel_name not in TRANSITIVE_RELATIONS:
            return 0
        facts = slot.get_all_facts()
        subj_to_objs: Dict[str, List[Tuple[str, float]]] = {}
        for s, o in facts:
            conf = slot.get_fact_confidence(s, o)
            subj_to_objs.setdefault(s, []).append((o, conf))

        new_count = 0
        for A, B_list in subj_to_objs.items():
            for (B, conf_ab) in B_list:
                if B in subj_to_objs:
                    for (C, conf_bc) in subj_to_objs[B]:
                        if C == A:
                            continue
                        if C in [obj for obj, _ in subj_to_objs.get(A, [])]:
                            continue
                        inferred_conf = conf_ab * conf_bc
                        existing_objs = slot.inferred_subj_to_objs.get(A, [])
                        existing_conf = next((c for o, c in existing_objs if o == C), None)
                        if existing_conf is not None and existing_conf >= inferred_conf:
                            continue
                        if slot.add_fact(A, C, ent_mgr, is_inferred=True, confidence=inferred_conf):
                            new_count += 1
        return new_count


# ----------------------------------------------------------------------------
# RuleMiner: Discovers heterogenous inference rules from facts
# ----------------------------------------------------------------------------
class RuleMiner:
    def __init__(self, min_confidence: float = RULE_MIN_CONFIDENCE):
        self.min_confidence = min_confidence
        self.rules: Dict[Tuple[str, str], Tuple[str, float]] = {}

    def mine(self, memory: 'AnalyticMemory') -> None:
        pair_counts: Dict[Tuple[str, str], int] = defaultdict(int)
        triple_counts: Dict[Tuple[str, str, str], int] = defaultdict(int)

        rel_names = list(memory.rel2slot.keys())
        for r1 in rel_names:
            slot1 = memory.slots[memory.rel2slot[r1]]
            facts1 = slot1.get_all_facts()
            for r2 in rel_names:
                slot2 = memory.slots[memory.rel2slot[r2]]
                facts2 = slot2.get_all_facts()
                b_to_c = defaultdict(list)
                for (b, c) in facts2:
                    b_to_c[b].append(c)
                for (a, b) in facts1:
                    if b in b_to_c:
                        for c in b_to_c[b]:
                            pair_counts[(r1, r2)] += 1
                            for r3 in rel_names:
                                slot3 = memory.slots[memory.rel2slot[r3]]
                                if any((s, o) == (a, c) for s, o in slot3.get_all_facts()):
                                    triple_counts[(r1, r2, r3)] += 1

        self.rules.clear()
        for (r1, r2, r3), count in triple_counts.items():
            total = pair_counts.get((r1, r2), 0)
            if total > 0:
                conf = count / total
                if conf >= self.min_confidence:
                    self.rules[(r1, r2)] = (r3, conf)

    def apply(self, memory: 'AnalyticMemory') -> int:
        new_facts = 0
        for (r1, r2), (r3, rule_conf) in self.rules.items():
            if r1 not in memory.rel2slot or r2 not in memory.rel2slot or r3 not in memory.rel2slot:
                continue
            slot1 = memory.slots[memory.rel2slot[r1]]
            slot2 = memory.slots[memory.rel2slot[r2]]
            slot3 = memory.slots[memory.rel2slot[r3]]
            facts1 = slot1.get_all_facts()
            facts2 = slot2.get_all_facts()
            b_to_c = defaultdict(list)
            for (b, c) in facts2:
                conf_b = slot2.get_fact_confidence(b, c)
                b_to_c[b].append((c, conf_b))
            for (a, b) in facts1:
                conf_a = slot1.get_fact_confidence(a, b)
                if b in b_to_c:
                    for (c, conf_b) in b_to_c[b]:
                        premise_conf = conf_a * conf_b
                        inferred_conf = premise_conf * rule_conf
                        existing = slot3.query(a, memory.ent_mgr)
                        if existing and existing[1] >= inferred_conf:
                            continue
                        if memory.add_fact(a, r3, c, inferred=True, confidence=inferred_conf):
                            slot3.update_performance(True, inferred_conf)
                            new_facts += 1
        return new_facts


# ----------------------------------------------------------------------------
# KnowledgeModule: Domain-isolated memory container
# ----------------------------------------------------------------------------
class KnowledgeModule:
    def __init__(self, module_id: str, dim: int, ent_mgr: Optional[EntityManager] = None, rel_mgr: Optional[RelationManager] = None):
        self.id = module_id
        self.dim = dim
        self.ent_mgr = ent_mgr or EntityManager(dim)
        self.rel_mgr = rel_mgr or RelationManager()
        self.slots: List[LogicSlot] = []
        self.rel2slot: Dict[str, int] = {}
        self.ilp = TransitiveReasoner()
        self.rule_miner = RuleMiner()
        self.rules_mined = False
        self.successful_sentences: List[str] = []
        self._domain_keywords: Set[str] = set()

    def add_domain_keyword(self, kw: str) -> None:
        self._domain_keywords.add(kw.lower())

    def is_relevant(self, context_text: str) -> bool:
        if not self._domain_keywords: return True
        ctx = context_text.lower()
        return any(kw in ctx for kw in self._domain_keywords)

    def add_fact(self, subj: str, rel: str, obj: str, inferred: bool = False, confidence: float = 1.0) -> bool:
        subj = normalize_entity_name(subj)
        obj = normalize_entity_name(obj)
        rel = rel.strip().lower()

        self.ent_mgr.get_or_create(subj)
        self.ent_mgr.get_or_create(obj)
        rel_idx = self.rel_mgr.get_or_create(rel)

        if rel not in self.rel2slot:
            slot_id = len(self.slots)
            self.slots.append(LogicSlot(slot_id, self.dim, rel_idx))
            self.rel2slot[rel] = slot_id

        return self.slots[self.rel2slot[rel]].add_fact(subj, obj, self.ent_mgr, is_inferred=inferred, confidence=confidence)

    def query_1hop(self, subj: str, rel: str) -> Optional[Tuple[str, float]]:
        subj = normalize_entity_name(subj)
        rel = rel.strip().lower()
        if rel not in self.rel2slot:
            return None
        return self.slots[self.rel2slot[rel]].query(subj, self.ent_mgr)

    def mine_rules(self) -> None:
        self.rule_miner.mine(self)
        self.rules_mined = True

    def apply_ilp(self) -> int:
        total = 0
        for rel, slot_idx in self.rel2slot.items():
            slot = self.slots[slot_idx]
            total += self.ilp.apply(slot, self.ent_mgr, rel)
        return total

    def apply_hetero_ilp(self) -> int:
        if not self.rules_mined: return 0
        return self.rule_miner.apply(self)

    def execute_plan(self, start_ent: str, rel_seq: List[str]) -> Optional[Tuple[str, List, float]]:
        current = normalize_entity_name(start_ent)
        trace = []
        total_conf = 1.0
        for rel in rel_seq:
            rel = rel.strip().lower()
            res = self.query_1hop(current, rel)
            if res is None: return None
            nxt, step_conf = res
            total_conf *= step_conf
            trace.append((current, rel, nxt, step_conf))
            current = nxt
        return current, trace, total_conf

    def reset(self) -> None:
        for slot in self.slots:
            slot.direct_facts.clear()
            slot.inferred_facts.clear()
            slot.direct_subj_to_obj.clear()
            slot.direct_obj_to_subj.clear()
            slot.inferred_subj_to_objs.clear()
            slot.inferred_confidences.clear()
            slot.subj_indices.clear()
            slot.obj_names.clear()
            slot.R = torch.randn_like(slot.R) * 0.01
            slot.velocity.zero_()
            slot.adaptation_factor = 1.0
            slot.performance_window.clear()
        self.rules_mined = False
        self.rule_miner.rules.clear()


# ----------------------------------------------------------------------------
# AnalyticMemory: Main memory system integrating all logic components (Facade Router)
# ----------------------------------------------------------------------------
class AnalyticMemory:
    def __init__(self, dim: int = D):
        self.dim = dim
        # ✅ Primary shared managers for backward compatibility with SEL 4/5/6/7/8
        self.ent_mgr = EntityManager(dim)
        self.rel_mgr = RelationManager()

        # ✅ Module routing architecture
        self.modules: Dict[str, KnowledgeModule] = {"default": KnowledgeModule("default", dim, self.ent_mgr, self.rel_mgr)}
        self.active_module_id: str = "default"

        # Facade properties pointing to active module
        self.slots = self.modules["default"].slots
        self.rel2slot = self.modules["default"].rel2slot
        self.ilp = self.modules["default"].ilp
        self.rule_miner = self.modules["default"].rule_miner
        self.rules_mined = self.modules["default"].rules_mined
        self.successful_sentences = self.modules["default"].successful_sentences

        self.error_log: deque = deque(maxlen=100)
        self.performance_history: deque = deque(maxlen=20)
        self.relation_error_counts: Dict[str, int] = defaultdict(int)
        self.relation_success_counts: Dict[str, int] = defaultdict(int)
        self.meta_controller = MetaController(self)

    def _switch_module(self, module_id: str) -> None:
        if module_id not in self.modules:
            self.modules[module_id] = KnowledgeModule(module_id, self.dim, self.ent_mgr, self.rel_mgr)
        self.active_module_id = module_id
        mod = self.modules[module_id]
        self.slots = mod.slots
        self.rel2slot = mod.rel2slot
        self.ilp = mod.ilp
        self.rule_miner = mod.rule_miner
        self.rules_mined = mod.rules_mined
        self.successful_sentences = mod.successful_sentences

    def add_fact(self, subj: str, rel: str, obj: str, inferred: bool = False, confidence: float = 1.0) -> bool:
        subj = normalize_entity_name(subj)
        rel = rel.strip().lower()
        obj = normalize_entity_name(obj)

        mod = self.modules[self.active_module_id]
        result = mod.add_fact(subj, rel, obj, inferred=inferred, confidence=confidence)
        if result:
            self.relation_success_counts[rel] += 1
        return result

    def mine_rules(self) -> None:
        self.modules[self.active_module_id].mine_rules()
        self.rules_mined = self.modules[self.active_module_id].rules_mined

    def apply_ilp(self) -> int:
        return self.modules[self.active_module_id].apply_ilp()

    def apply_hetero_ilp(self) -> int:
        if not self.modules[self.active_module_id].rules_mined:
            return 0
        return self.modules[self.active_module_id].rule_miner.apply(self)

    def query_1hop(self, subj: str, rel: str) -> Optional[Tuple[str, float]]:
        subj = normalize_entity_name(subj)
        rel = rel.strip().lower()
        mod = self.modules[self.active_module_id]
        if rel not in mod.rel2slot:
            return None
        return mod.slots[mod.rel2slot[rel]].query(subj, mod.ent_mgr)

    def query_1hop_inverse(self, obj: str, rel: str) -> Optional[Tuple[str, float]]:
        obj = normalize_entity_name(obj)
        rel = rel.strip().lower()
        mod = self.modules[self.active_module_id]
        if rel not in mod.rel2slot:
            return None
        return mod.slots[mod.rel2slot[rel]].inverse_query(obj, mod.ent_mgr)

    def execute_plan(self, start_ent: str, rel_seq: List[str]) -> Optional[Tuple[str, List, float]]:
        return self.modules[self.active_module_id].execute_plan(start_ent, rel_seq)

    def record_error(self, question: str, expected: str, actual: str, confidence: float, relation: str = None) -> None:
        error_entry = {"question": question, "expected": expected, "actual": actual, "confidence": confidence, "relation": relation, "timestamp": len(self.error_log)}
        self.error_log.append(error_entry)
        if relation: self.relation_error_counts[relation] += 1

    def adapt_based_on_errors(self) -> None:
        global GLOBAL_CONFIDENCE_THRESHOLD
        if len(self.error_log) < 5: return
        recent_errors = list(self.error_log)[-10:] if len(self.error_log) >= 10 else list(self.error_log)
        correct_predictions = sum(1 for err in recent_errors if err["expected"] == err["actual"])
        recent_accuracy = correct_predictions / len(recent_errors) if recent_errors else 0
        self.performance_history.append(recent_accuracy)
        if len(self.performance_history) > 1:
            current_acc = self.performance_history[-1]
            prev_acc = self.performance_history[-2]
            if current_acc < prev_acc - 0.1:
                GLOBAL_CONFIDENCE_THRESHOLD = min(0.9, GLOBAL_CONFIDENCE_THRESHOLD + 0.05)
                logger.info(f"🔧 Self-adaptation: confidence threshold dinaikkan ke {GLOBAL_CONFIDENCE_THRESHOLD}")
            elif current_acc > prev_acc + 0.1:
                GLOBAL_CONFIDENCE_THRESHOLD = max(0.3, GLOBAL_CONFIDENCE_THRESHOLD - 0.02)
                logger.info(f"🔧 Self-adaptation: confidence threshold diturunkan ke {GLOBAL_CONFIDENCE_THRESHOLD}")

    def analyze_relation_performance(self) -> Dict[str, Dict]:
        analysis = {}
        for rel in self.rel2slot:
            total_attempts = self.relation_success_counts[rel] + self.relation_error_counts[rel]
            if total_attempts > 0:
                accuracy = self.relation_success_counts[rel] / total_attempts
                analysis[rel] = {'accuracy': accuracy, 'success_count': self.relation_success_counts[rel], 'error_count': self.relation_error_counts[rel], 'total_attempts': total_attempts}
        return analysis

    def get_plan_vector(self, rel_seq: List[str], rel_embeddings: nn.Embedding) -> torch.Tensor:
        if not rel_seq: return torch.zeros(self.dim, device=device)
        idxs = [self.rel_mgr.get_idx(r) for r in rel_seq if self.rel_mgr.get_idx(r) is not None]
        if not idxs: return torch.zeros(self.dim, device=device)
        idxs = torch.tensor(idxs, device=device)
        embs = rel_embeddings(idxs)
        plan_vec = embs.mean(dim=0)
        return F.normalize(plan_vec, dim=0)

    def reset(self) -> None:
        for mod in self.modules.values(): mod.reset()
        self._switch_module("default")
        self.rules_mined = False
        self.error_log.clear()
        self.performance_history.clear()
        self.relation_error_counts.clear()
        self.relation_success_counts.clear()
        self.meta_controller = MetaController(self)


# ----------------------------------------------------------------------------
# MetaController: Adaptive parameter tuning based on runtime metrics
# ----------------------------------------------------------------------------
class MetaController:
    def __init__(self, memory: 'AnalyticMemory'):
        self.memory = memory
        self.params = {"retrieval_top_k": 3, "sim_threshold": 0.9, "retrieval_conf_threshold": 0.3, "ilp_confidence_base": 0.8, "neural_extraction_conf": 0.9, "max_plan_len": 5, "max_retrieval_retries": 2}
        self.metric_history = deque(maxlen=100)
        self.query_counter = 0
        self.ADAPT_INTERVAL = 20

    def record_query_metrics(self, metrics: Dict) -> None:
        self.metric_history.append(metrics)
        self.query_counter += 1
        if self.query_counter % self.ADAPT_INTERVAL == 0:
            self._adapt_all()

    def _adapt_all(self) -> None:
        self._adapt_retrieval_k()
        self._adapt_sim_threshold()
        self._adapt_retrieval_conf_threshold()
        self._adapt_ilp_confidence()
        self._adapt_max_plan_len()

    def _adapt_retrieval_k(self) -> None:
        recent_sims = [m.get("retrieved_sims", []) for m in self.metric_history if "retrieved_sims" in m]
        if not recent_sims: return
        avg_std = np.mean([np.std(s) if len(s) > 1 else 0 for s in recent_sims if len(s) > 1])
        if avg_std < 0.05: self.params["retrieval_top_k"] = min(10, self.params["retrieval_top_k"] + 1)
        elif avg_std > 0.2: self.params["retrieval_top_k"] = max(2, self.params["retrieval_top_k"] - 1)

    def _adapt_sim_threshold(self) -> None:
        fp = sum(1 for m in self.metric_history if m.get("is_false_positive", False))
        fn = sum(1 for m in self.metric_history if m.get("is_false_negative", False))
        total = len(self.metric_history)
        if total == 0: return
        fp_rate, fn_rate = fp / total, fn / total
        if fp_rate > 0.2: self.params["sim_threshold"] = min(0.99, self.params["sim_threshold"] + 0.02)
        elif fn_rate > 0.2: self.params["sim_threshold"] = max(0.7, self.params["sim_threshold"] - 0.02)

    def _adapt_retrieval_conf_threshold(self) -> None:
        samples = [(m["fallback_sim"], m["is_correct"]) for m in self.metric_history if "fallback_sim" in m]
        if len(samples) < 10: return
        best_thresh = 0.3; best_f1 = 0.0
        for th in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
            tp = sum(1 for s, c in samples if s >= th and c)
            fp = sum(1 for s, c in samples if s >= th and not c)
            fn = sum(1 for s, c in samples if s < th and c)
            prec = tp / (tp+fp) if (tp+fp) >0 else 0
            rec = tp / (tp+fn) if (tp+fn) >0 else 0
            f1 = 2*prec*rec/(prec+rec) if (prec+rec) >0 else 0
            if f1 > best_f1: best_f1 = f1; best_thresh = th
        self.params["retrieval_conf_threshold"] = best_thresh

    def _adapt_ilp_confidence(self) -> None:
        short = [m for m in self.metric_history if m.get("chain_depth",0)==2]
        long = [m for m in self.metric_history if m.get("chain_depth",0) >=3]
        acc_short = sum(1 for m in short if m["is_correct"]) / len(short) if short else 1.0
        acc_long = sum(1 for m in long if m["is_correct"]) / len(long) if long else 1.0
        if acc_long < acc_short * 0.7: self.params["ilp_confidence_base"] = max(0.5, self.params["ilp_confidence_base"] - 0.05)
        elif acc_long > acc_short * 0.9: self.params["ilp_confidence_base"] = min(0.95, self.params["ilp_confidence_base"] + 0.02)

    def _adapt_max_plan_len(self) -> None:
        success = sum(1 for m in self.metric_history if m.get("inverse_fallback_used") and m.get("is_correct"))
        total = sum(1 for m in self.metric_history if m.get("inverse_fallback_used"))
        if total > 5 and success / total > 0.6: self.params["max_plan_len"] = min(8, self.params["max_plan_len"] + 1)


# ----------------------------------------------------------------------------
# generate_plans_beam: Autonomous symbolic planning with rule-aware expansion
# ----------------------------------------------------------------------------
def generate_plans_beam(
    mem: AnalyticMemory,
    anchor: str,
    target_hint: Optional[str] = None,
    beam_width: int = 3,
    max_depth: int = MAX_PLAN_LEN
) -> List[Tuple[List[str], float]]:
    # State: (current_entity, plan_path, cumulative_conf, visited_set)
    beam = [(anchor, [], 1.0, {anchor})]
    final_plans = []

    for depth in range(max_depth):
        next_beam = []
        for curr_ent, curr_plan, curr_conf, visited in beam:
            # 1. Direct relation expansion
            for rel, slot_idx in mem.rel2slot.items():
                nxt_ent = None
                step_conf = 0.0
                res = mem.query_1hop(curr_ent, rel)
                if res:
                    nxt_ent, step_conf = res
                else:
                    # Try inverse
                    inv_res = mem.query_1hop_inverse(curr_ent, rel)
                    if inv_res:
                        nxt_ent, step_conf = inv_res

                if nxt_ent and nxt_ent not in visited:
                    new_conf = curr_conf * step_conf
                    new_plan = curr_plan + [rel]
                    new_visited = visited | {nxt_ent}

                    if target_hint and nxt_ent == target_hint:
                        final_plans.append((new_plan, new_conf))
                    else:
                        next_beam.append((nxt_ent, new_plan, new_conf, new_visited))

            # 2. Heterogeneous rule expansion (multi-hop virtual jump)
            for (r1, r2), (r3, rule_conf) in mem.rule_miner.rules.items():
                if r1 in curr_plan: continue # prevent rule loops
                res = mem.query_1hop(curr_ent, r1)
                if res:
                    mid, c1 = res
                    res2 = mem.query_1hop(mid, r2)
                    if res2 and mid not in visited:
                        _, c2 = res2
                        pred_conf = curr_conf * c1 * c2 * rule_conf
                        next_beam.append((curr_ent, curr_plan + [r1, r2, r3], pred_conf, visited))

        # Pruning
        next_beam.sort(key=lambda x: x[2], reverse=True)
        beam = next_beam[:beam_width]
        if not beam: break

    if final_plans:
        final_plans.sort(key=lambda x: x[1], reverse=True)
        return final_plans[:beam_width]
    return [(p, c) for _, p, c, _ in beam]

In [62]:
# @title ✅ SEL 4: Neural Components (Encoder, Dataset, Retriever, Training)
# ============================================================================

# ----------------------------------------------------------------------------
# IntuitionEncoderConditioned: FIXED with proper dynamic vocab expansion
# ----------------------------------------------------------------------------
class IntuitionEncoderConditioned(nn.Module):
    def __init__(self, tokenizer: DynamicTokenizer, dim: int,
                 ent_mgr: EntityManager, rel_mgr: RelationManager,
                 max_len: int = MAX_SEQ_LEN):
        super().__init__()
        self.tokenizer = tokenizer
        self.ent_mgr = ent_mgr
        self.rel_mgr = rel_mgr
        self.dim = dim
        self.max_len = max_len

        vocab_size = len(tokenizer)
        num_entities = len(ent_mgr.entities)
        num_relations = len(rel_mgr.relations)

        self.token_embed = nn.Embedding(vocab_size, dim)
        self.pos_embed = nn.Embedding(max_len, dim)
        self.rel_embed = nn.Embedding(num_relations, dim)
        self.neural_ent_embed = nn.Embedding(num_entities, dim)

        encoder_layer = nn.TransformerEncoderLayer(d_model=dim, nhead=4, dim_feedforward=dim*2, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.mediator_C = nn.Parameter(torch.randn(dim) * 0.1)
        self.rel_classifier = nn.Linear(dim, num_relations)
        self.plan_gate = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.Sigmoid()
        )

        self.performance_score = 1.0
        self.learning_rate_adjustment = 1.0
        self.base_lr = NEURAL_LR

        # Register callbacks for dynamic expansion
        ent_mgr.register_new_entity_callback(lambda name, idx: self._on_new_entity(name, idx))
        rel_mgr.register_new_relation_callback(lambda name, idx: self._on_new_relation(name, idx))

    def _on_new_entity(self, name: str, idx: int) -> None:
        """Called when new entity is registered in ent_mgr."""
        self._expand_embedding('neural_ent_embed', idx + 1)

    def _on_new_relation(self, name: str, idx: int) -> None:
        """Called when new relation is registered in rel_mgr."""
        self._expand_embedding('rel_embed', idx + 1)
        # Also expand classifier output
        self._expand_classifier(idx + 1)

    def _expand_embedding(self, attr_name: str, new_size: int) -> None:
        """Safely expand an embedding layer to new_size."""
        embed = getattr(self, attr_name)
        old_size = embed.num_embeddings
        if new_size <= old_size:
            return  # Already big enough

        # Create new embedding with expanded size
        new_embed = nn.Embedding(new_size, embed.embedding_dim).to(embed.weight.device)
        # Copy old weights
        new_embed.weight.data[:old_size] = embed.weight.data
        # Initialize new weights randomly
        if new_size > old_size:
            new_embed.weight.data[old_size:] = torch.randn(
                new_size - old_size, embed.embedding_dim, device=embed.weight.device
            ) * 0.01
        setattr(self, attr_name, new_embed)

    def _expand_classifier(self, new_num_relations: int) -> None:
        """Expand relation classifier output layer."""
        old_out = self.rel_classifier.out_features
        if new_num_relations <= old_out:
            return

        new_linear = nn.Linear(self.dim, new_num_relations).to(self.rel_classifier.weight.device)
        # Copy old weights and biases
        new_linear.weight.data[:old_out] = self.rel_classifier.weight.data
        new_linear.bias.data[:old_out] = self.rel_classifier.bias.data
        # Initialize new outputs
        if new_num_relations > old_out:
            new_linear.weight.data[old_out:] = torch.randn(
                new_num_relations - old_out, self.dim, device=self.rel_classifier.weight.device
            ) * 0.01
            new_linear.bias.data[old_out:] = torch.zeros(
                new_num_relations - old_out, device=self.rel_classifier.bias.device
            )
        self.rel_classifier = new_linear

    def sync_tokenizer_vocab(self) -> None:
        """Ensure token_embed matches tokenizer vocabulary size."""
        vocab_size = len(self.tokenizer.word2idx)
        current_size = self.token_embed.num_embeddings
        if vocab_size > current_size:
            self._expand_embedding('token_embed', vocab_size)
            logger.info(f"🔧 Expanded token_embed: {current_size} → {vocab_size}")

    def forward(self, input_ids: torch.Tensor, attention_mask: Optional[torch.Tensor] = None,
                plan_vec: Optional[torch.Tensor] = None) -> Dict[str, torch.Tensor]:
        # Fixed: Sync vocab before forward pass (defensive)
        self.sync_tokenizer_vocab()

        B, T = input_ids.shape
        device = input_ids.device
        pos = torch.arange(T, device=device).unsqueeze(0).expand(B, T)
        x = self.token_embed(input_ids) + self.pos_embed(pos)

        mask = ~attention_mask.bool() if attention_mask is not None else None
        x = self.transformer(x, src_key_padding_mask=mask)

        if attention_mask is not None:
            expanded_mask = attention_mask.unsqueeze(-1).float()
            pooled = (x * expanded_mask).sum(dim=1) / expanded_mask.sum(dim=1).clamp(min=1e-9)
        else:
            pooled = x.mean(dim=1)

        cond = plan_vec if plan_vec is not None else self.mediator_C.unsqueeze(0).expand(B, -1)
        gate_input = torch.cat([pooled, cond], dim=-1)
        gate_val = self.plan_gate(gate_input)
        fused = gate_val * pooled + (1 - gate_val) * cond

        rel_logits = self.rel_classifier(fused)
        fused_norm = F.normalize(fused, p=2, dim=-1)

        return {
            "pooled": pooled,
            "fused": fused,
            "fused_norm": fused_norm,
            "rel_logits": rel_logits
        }

    # ... (adjust_learning_rate tetap sama) ...

    def adjust_learning_rate(self, optimizer, performance_metric: float) -> None:
        if performance_metric < 0.7:
            self.learning_rate_adjustment = min(2.0, self.learning_rate_adjustment * 1.1)
        elif performance_metric > 0.9:
            self.learning_rate_adjustment = max(0.1, self.learning_rate_adjustment * 0.95)
        for param_group in optimizer.param_groups:
            param_group['lr'] = self.base_lr * self.learning_rate_adjustment


# ----------------------------------------------------------------------------
# ReasoningDataset: Dataset wrapper for training data
# ----------------------------------------------------------------------------
class ReasoningDataset(Dataset):
    def __init__(self, data, tokenizer: DynamicTokenizer, ent_mgr: EntityManager,
                 rel_mgr: RelationManager, max_len=MAX_SEQ_LEN, max_plan_len=MAX_PLAN_LEN):
        self.data = data
        self.tokenizer = tokenizer
        self.ent_mgr = ent_mgr
        self.rel_mgr = rel_mgr
        self.max_len = max_len
        self.max_plan_len = max_plan_len

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        item = self.data[idx]
        sent_ids = self.tokenizer.tokenize(item["sentence"])[:self.max_len]
        # ✅ FIX: Gunakan token "<pad>" yang selaras dengan DynamicTokenizer (SEL 2)
        pad_id = self.tokenizer.word2idx["<pad>"]
        sent_ids += [pad_id] * (self.max_len - len(sent_ids))
        sent_mask = [1] * min(len(self.tokenizer.tokenize(item["sentence"])), self.max_len) + [0] * (self.max_len - min(len(self.tokenizer.tokenize(item["sentence"])), self.max_len))

        answer = item["answer"]
        answer_idx = self.ent_mgr.entity2idx.get(answer, 0)

        plan = item["plan"]
        rel_idx = self.rel_mgr.get_idx(plan[0]) if plan else 0
        plan_indices = [self.rel_mgr.get_idx(r) if self.rel_mgr.get_idx(r) is not None else 0 for r in plan]
        plan_tensor = torch.tensor(plan_indices + [0] * (self.max_plan_len - len(plan_indices)), dtype=torch.long)
        plan_mask = torch.tensor([1] * len(plan_indices) + [0] * (self.max_plan_len - len(plan_indices)), dtype=torch.bool)

        return {
            "sent_ids": torch.tensor(sent_ids, dtype=torch.long),
            "sent_mask": torch.tensor(sent_mask, dtype=torch.float),
            "rel_idx": torch.tensor(rel_idx, dtype=torch.long),
            "obj_idx": torch.tensor(answer_idx, dtype=torch.long),
            "plan_tensor": plan_tensor,
            "plan_mask": plan_mask,
        }


# ----------------------------------------------------------------------------
# Training Function
# ----------------------------------------------------------------------------
def train_extractor(model: IntuitionEncoderConditioned, dataloader: DataLoader,
                    optimizer, epochs: int, lambda_final=ALIGNMENT_LAMBDA) -> None:
    model.train()
    for epoch in range(epochs):
        progress = min(1.0, epoch / (epochs * 0.66))
        current_lambda = lambda_final * progress
        total_loss, total_rel, total_align = 0.0, 0.0, 0.0
        for batch in dataloader:
            sent_ids = batch["sent_ids"].to(device)
            sent_mask = batch["sent_mask"].to(device)
            rel_idx = batch["rel_idx"].to(device)
            obj_idx = batch["obj_idx"].to(device)
            plan_tensors = batch["plan_tensor"].to(device)
            plan_masks = batch["plan_mask"].to(device)

            batch_plan_vecs = []
            for i in range(plan_tensors.size(0)):
                valid = plan_tensors[i][plan_masks[i]]
                if valid.size(0) > 0:
                    pv = model.rel_embed(valid).mean(dim=0)
                else:
                    pv = torch.zeros(model.dim, device=device)
                batch_plan_vecs.append(F.normalize(pv, dim=-1))
            plan_vec_batch = torch.stack(batch_plan_vecs)

            out = model(sent_ids, sent_mask, plan_vec=plan_vec_batch)

            loss_rel = F.cross_entropy(out["rel_logits"], rel_idx)
            target_emb = model.neural_ent_embed(obj_idx)
            target_emb_norm = F.normalize(target_emb, p=2, dim=-1)
            cos_sim = (out["fused_norm"] * target_emb_norm).sum(dim=-1)
            loss_align = (1.0 - cos_sim).mean()

            loss = loss_rel + current_lambda * loss_align
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_rel += loss_rel.item()
            total_align += loss_align.item()

        if (epoch+1) % 10 == 0:
            logger.info(f"  Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataloader):.4f}  "
                  f"(Rel: {total_rel/len(dataloader):.4f}, Align: {total_align/len(dataloader):.4f})")

# ----------------------------------------------------------------------------
# NeuralRetriever: Embedding-based sentence retrieval (FIXED)
# ----------------------------------------------------------------------------
class NeuralRetriever:
    def __init__(self, model: Optional[IntuitionEncoderConditioned],
                 corpus: List[str],
                 tokenizer: Optional[DynamicTokenizer]):
        self.model = model
        self.corpus = corpus
        self.tokenizer = tokenizer
        self.corpus_embeddings: Optional[torch.Tensor] = None

        # Fixed: Only encode if model & tokenizer are available
        if self.model is not None and self.tokenizer is not None and self.corpus:
            self._encode_corpus()

    def _encode_corpus(self) -> None:
        """Encode corpus into embeddings. Safe to call multiple times."""
        if self.model is None or self.tokenizer is None or not self.corpus:
            logger.warning("⚠️ Cannot encode corpus: model/tokenizer/corpus not ready")
            return

        self.model.eval()
        embs = []
        with torch.no_grad():
            for sent in self.corpus:
                ids = torch.tensor(self.tokenizer.tokenize(sent), device=device).unsqueeze(0)
                mask = torch.ones_like(ids).float()
                out = self.model(ids, mask)
                embs.append(F.normalize(out["pooled"], p=2, dim=-1))
        self.corpus_embeddings = torch.cat(embs, dim=0)
        logger.info(f"📊 Encoded {len(self.corpus)} sentences into corpus embeddings")

    def retrieve_with_similarities(self, query_fused: torch.Tensor, top_k: int = RETRIEVAL_TOP_K) -> Tuple[List[str], List[float]]:
        if self.corpus_embeddings is None or len(self.corpus) == 0:
            return [], []
        query_norm = F.normalize(query_fused, p=2, dim=-1)
        sims = torch.mv(self.corpus_embeddings, query_norm.squeeze(0))
        top_values, top_indices = torch.topk(sims, min(top_k, len(self.corpus)))
        retrieved_sentences = [self.corpus[i] for i in top_indices.tolist()]
        retrieved_similarities = top_values.tolist()
        return retrieved_sentences, retrieved_similarities

    def set_model_and_tokenize(self, model: IntuitionEncoderConditioned, tokenizer: DynamicTokenizer) -> None:
        """Set model/tokenizer after initialization and encode corpus if needed."""
        self.model = model
        self.tokenizer = tokenizer
        if self.corpus and self.corpus_embeddings is None:
            self._encode_corpus()

In [63]:
# @title ✅ SEL 5: QA & Learning Functions (Extraction, Interactive QA, Active Learning) - FIXED
# ============================================================================

# ----------------------------------------------------------------------------
# Global irregular verbs mapping
# ----------------------------------------------------------------------------
irregular_verbs = {
    "caught": "catch", "ate": "eat", "gave": "give", "bought": "buy",
    "saw": "see", "was": "be", "were": "be", "is": "be", "are": "be",
    "chased": "chase", "read": "read"
}


# ----------------------------------------------------------------------------
# extract_triplets_strict: spaCy dependency-based extraction with confidence scoring
# ----------------------------------------------------------------------------
def extract_triplets_strict(sentence: str, mem: 'AnalyticMemory', confidence: float = None) -> Optional[Tuple[str, str, str, float]]:
    """
    Extract subject-relation-object triplets using spaCy dependency parsing.
    Confidence is computed based on dependency role accuracy.
    New entities and relations are auto-registered.
    """
    doc = nlp(sentence)

    # 1. Find verb token as relation
    rel_token = None
    for token in doc:
        if token.pos_ == "VERB":
            lemma = token.lemma_.lower()
            mem.rel_mgr.get_or_create(lemma)
            rel_token = token
            break
    if rel_token is None:
        return None

    rel_name = rel_token.lemma_.lower()

    # 2. Find subject via dependency roles (nsubj/nsubjpass)
    subj_token = None
    for child in rel_token.children:
        if child.dep_ in ("nsubj", "nsubjpass"):
            subj_token = child
            break

    # Limited fallback: NOUN/PROPN directly left of verb
    if subj_token is None:
        for token in reversed(list(doc)[:rel_token.i]):
            if token.pos_ in ("NOUN", "PROPN") and not token.is_stop:
                subj_token = token
                break

    # 3. Find object via dependency roles (dobj/attr/pobj)
    obj_token = None
    for child in rel_token.children:
        if child.dep_ in ("dobj", "attr"):
            obj_token = child
            break
        # Handle prepositional dative: "give to Mary"
        if child.dep_ == "prep" and child.text.lower() == "to":
            for grandchild in child.children:
                if grandchild.dep_ == "pobj":
                    obj_token = grandchild
                    break
            if obj_token:
                break

    # Limited fallback: NOUN/PROPN first right of verb
    if obj_token is None:
        for token in list(doc)[rel_token.i+1:]:
            if token.pos_ in ("NOUN", "PROPN") and not token.is_stop:
                obj_token = token
                break

    # Fail if either not found (no fake triplets)
    if subj_token is None or obj_token is None:
        return None

    subj = subj_token.text.lower()
    obj = obj_token.text.lower()

    # 4. Compute confidence based on dependency roles
    dep_subj = subj_token.dep_ in ("nsubj", "nsubjpass")
    dep_obj = obj_token.dep_ in ("dobj", "attr", "pobj")
    if dep_subj and dep_obj:
        conf = 1.0
    elif dep_subj or dep_obj:
        conf = 0.8
    else:
        conf = 0.6

    # Use provided confidence if specified and higher
    if confidence is not None:
        conf = max(conf, confidence)

    # 5. Auto-register entities
    mem.ent_mgr.get_or_create(subj)
    mem.ent_mgr.get_or_create(obj)

    # Simpan kalimat jika confidence >= 0.9 dan mem memiliki atribut successful_sentences
    if conf >= 0.9 and hasattr(mem, 'successful_sentences'):
        mem.successful_sentences.append(sentence)

    return subj, rel_name, obj, conf


def extract_triplets_from_text(model, text: str, mem: 'AnalyticMemory') -> List[Tuple[str, str, str, float]]:
    sentences = text.split(" . ")
    triplets = []
    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue
        if not sent.endswith("."):
            sent += " . "
        res = extract_triplets_strict(sent, mem)
        if res:
            triplets.append(res)
    return triplets


# ----------------------------------------------------------------------------
# Explainer & Active Learning Utilities
# ----------------------------------------------------------------------------
class Explainer:
    @staticmethod
    def format_trace(trace: List[Tuple[str, str, str, float]], final_answer: str, final_conf: float) -> str:
        if not trace:
            return f"Jawaban: {final_answer} (confidence={final_conf:.2f}) [tidak ada jejak]"
        lines = ["📋 Penjelasan Langkah demi Langkah"]
        for i, (subj, rel, obj, conf) in enumerate(trace, 1):
            lines.append(f"  {i}. {subj} --[{rel}]--> {obj} (confidence={conf:.2f})")
        lines.append(f"\n✅ Jawaban Akhir : {final_answer} (total confidence={final_conf:.2f})")
        return "\n".join(lines)


def ask_user_confirmation(question: str, proposed_answer: str, confidence: float) -> bool:
    print(f"\n❓ Sistem tidak yakin (conf={confidence:.2f}). ")
    print(f"   Pertanyaan: {question} ")
    print(f"   Dugaan jawaban: {proposed_answer} ")
    response = input("   Apakah jawaban ini benar? (y/n): ").strip().lower()
    return response == 'y'


# ----------------------------------------------------------------------------
# Interactive QA with Active Learning
# ----------------------------------------------------------------------------
def interactive_qa_with_active_learning(
    question: str,
    plan: List[str],
    expected_answer: Optional[str],
    mem: 'AnalyticMemory',
    model: 'IntuitionEncoderConditioned',
    retriever: 'NeuralRetriever',
    tokenizer: 'DynamicTokenizer',
    conf_threshold: float = GLOBAL_CONFIDENCE_THRESHOLD,
    verbose: bool = True,
    interactive: bool = True
) -> Dict[str, Any]:
    # ========================================================================
    # 🧠 AUTONOMOUS PLANNING INTEGRATION
    # ========================================================================
    if plan is None:
        plan = []

    # ✅ FIX: Definisikan known_entities di awal agar selalu tersedia
    tokens = question.lower().replace("?", " ").split()
    known_entities = set(mem.ent_mgr.entities)

    if not plan:
        q_tokens = tokens  # sudah didefinisikan di atas
        known_rels = mem.rel_mgr.relations

        # Deteksi anchor untuk beam search
        anchor_ent = None
        for tok in reversed(q_tokens):
            if tok in known_entities:
                anchor_ent = tok
                break

        # Generate rencana via beam search simbolik
        if anchor_ent:
            candidate_plans = generate_plans_beam(mem, anchor_ent, beam_width=3, max_depth=MAX_PLAN_LEN)
            if candidate_plans:
                plan, _ = candidate_plans[0]

        # Fallback ke heuristic keyword jika beam search kosong
        if not plan:
            for tok in q_tokens:
                base = irregular_verbs.get(tok, tok)
                if base in known_rels:
                    plan = [base]
                    break
                if tok in known_rels:
                    plan = [tok]
                    break
    # ========================================================================

    plan_vec = mem.get_plan_vector(plan, model.rel_embed)
    q_ids = torch.tensor(tokenizer.tokenize(question), device=device).unsqueeze(0)
    q_mask = torch.ones_like(q_ids).float()
    with torch.no_grad():
        out = model(q_ids, q_mask, plan_vec=plan_vec.unsqueeze(0) if plan_vec.dim() == 1 else plan_vec)

    is_who_question = "who" in question.lower()

    anchor_ent = None
    if is_who_question:
        if "to" in tokens:
            try:
                to_idx = tokens.index("to")
                for j in range(to_idx + 1, len(tokens)):
                    if tokens[j] in known_entities:
                        anchor_ent = tokens[j]
                        break
            except ValueError:
                pass
        if anchor_ent is None:
            for tok in reversed(tokens):
                if tok in known_entities:
                    anchor_ent = tok
                    break
    else:
        for tok in tokens:
            if tok in known_entities:
                anchor_ent = tok
                break

    if anchor_ent is None:
        return {"correct": False, "answer": "None", "confidence": 0.0, "explanation": "Entitas tidak ditemukan."}

    def execute_with_inverse_fallback(start_ent: str, rel_seq: List[str]):
        res = mem.execute_plan(start_ent, rel_seq)
        if res is not None:
            return res
        if len(rel_seq) > 0:
            first_rel = rel_seq[0]
            inv = mem.query_1hop_inverse(start_ent, first_rel)
            if inv:
                new_start, _ = inv
                return mem.execute_plan(new_start, rel_seq)
        return None

    def compute_result():
        if is_who_question and len(plan) == 1:
            rel = plan[0]
            res = mem.query_1hop_inverse(anchor_ent, rel)
            if res:
                subj, conf = res
                return subj, [], conf
            return None
        else:
            return execute_with_inverse_fallback(anchor_ent, plan)

    result = compute_result()
    final_conf = 0.0
    trace = []
    max_retries = mem.meta_controller.params["max_retrieval_retries"]
    retry_count = 0
    retrieved_sims_list = []

    while result is None and retry_count < max_retries:
        if verbose:
            print(f"    🔄 Retrieval attempt {retry_count+1}...")
        retrieved_sents, retrieved_sims = retriever.retrieve_with_similarities(
            out["fused_norm"], top_k=mem.meta_controller.params["retrieval_top_k"])
        retrieved_sims_list.extend(retrieved_sims)
        any_added = False
        for sent in retrieved_sents:
            triplets = extract_triplets_from_text(model, sent, mem)
            for subj, rel, obj, conf in triplets:
                if mem.add_fact(subj, rel, obj, inferred=False, confidence=conf):
                    any_added = True
                    if verbose:
                        print(f"       Added fact: {subj} --[{rel}]--> {obj} (conf={conf:.2f})")
        if not any_added:
            break
        mem.apply_ilp()
        mem.apply_hetero_ilp()
        result = compute_result()
        retry_count += 1

    if result:
        final, trace, final_conf = result if isinstance(result, tuple) and len(result) == 3 else (result[0], [], 1.0)
        is_correct = (expected_answer is None) or (final == expected_answer)
        if expected_answer is not None:
            relation_for_tracking = plan[0] if plan else None
            mem.record_error(question, expected_answer, final, final_conf, relation_for_tracking)
            mem.adapt_based_on_errors()

        if final_conf < conf_threshold and interactive:
            print(Explainer.format_trace(trace, final, final_conf))
            if ask_user_confirmation(question, final, final_conf):
                if is_who_question and len(plan) == 1:
                    rel = plan[0]
                    mem.add_fact(final, rel, anchor_ent, inferred=False, confidence=1.0)
                    print(f"   ✅ Belajar: menambahkan fakta {final} --[{rel}]--> {anchor_ent}")
                else:
                    if plan:
                        mem.add_fact(anchor_ent, plan[0], final, inferred=False, confidence=1.0)
                        print(f"   ✅ Belajar: menambahkan fakta {anchor_ent} --[{plan[0]}]--> {final}")
                final_conf = 1.0
                return {
                    "correct": True,
                    "answer": final,
                    "confidence": final_conf,
                    "explanation": Explainer.format_trace(trace, final, final_conf) + "\n   (Dikonfirmasi oleh pengguna)",
                    "trace": trace
                }
            else:
                return {
                    "correct": False,
                    "answer": "UNKNOWN (ditolak pengguna)",
                    "confidence": final_conf,
                    "explanation": Explainer.format_trace(trace, final, final_conf) + "\n   (Ditolak oleh pengguna)",
                    "trace": trace
                }
        else:
            explanation = Explainer.format_trace(trace, final, final_conf)
            plan_depth = len(plan) if plan else 0
            metrics = {
                "is_correct": is_correct,
                "confidence": final_conf,
                "chain_depth": plan_depth,
                "inverse_fallback_used": (is_who_question and len(plan) == 1 and mem.query_1hop_inverse(anchor_ent, plan[0]) is not None),
                "retrieved_sims": retrieved_sims_list,
                "is_false_positive": final_conf > conf_threshold and not is_correct,
                "is_false_negative": final_conf < conf_threshold and is_correct,
                "fallback_sim": final_conf if len(trace) == 0 else 0
            }
            mem.meta_controller.record_query_metrics(metrics)
            return {
                "correct": is_correct,
                "answer": final,
                "confidence": final_conf,
                "explanation": explanation,
                "trace": trace
            }
    else:
        if interactive:
            print(f"\n❓ Sistem tidak dapat menemukan jawaban untuk: {question}")
        return {
            "correct": False,
            "answer": "UNKNOWN",
            "confidence": 0.0,
            "explanation": "Tidak dapat menemukan jawaban.",
            "trace": []
        }


# ----------------------------------------------------------------------------
# Dataset Generator (Dynamic)
# ----------------------------------------------------------------------------
def generate_multi_hop_dataset(ent_mgr: EntityManager, rel_mgr: RelationManager,
                               num_single: int = 200, num_double: int = 100, num_triple: int = 50) -> List[Dict]:
    if len(ent_mgr.entities) == 0:
        initial_entities = ["dog", "cat", "mouse", "cheese", "kitchen", "john", "book", "mary",
                             "table", "room", "alice", "flowers", "park", "car", "house", "ball",
                             "child", "mother", "father", "tree"]
        for e in initial_entities:
            ent_mgr.get_or_create(e)
    if len(rel_mgr.relations) == 0:
        initial_relations = ["chase", "catch", "eat", "be", "give", "read", "buy", "see"]
        for r in initial_relations:
            rel_mgr.get_or_create(r)

    entities = ent_mgr.entities
    relations = rel_mgr.relations

    data = []
    forced_single = [
        ("the dog chased the cat .", ("dog", "chase", "cat")),
        ("the cat caught the mouse .", ("cat", "catch", "mouse")),
        ("the mouse ate the cheese .", ("mouse", "eat", "cheese")),
        ("the cheese was in the kitchen .", ("cheese", "be", "kitchen")),
        ("john gave the book to mary .", ("john", "give", "mary")),
        ("mary read the book in the park .", ("mary", "read", "book")),
    ]
    for sent, (subj, rel, obj) in forced_single:
        for _ in range(10):
            data.append({"sentence": sent, "triplets": [(subj, rel, obj)], "plan": [rel], "answer": obj})

    for _ in range(num_single):
        subj, obj = random.sample(entities, 2)
        rel = random.choice(relations)
        sent = f"the {subj} {rel} the {obj} ."
        data.append({"sentence": sent, "triplets": [(subj, rel, obj)], "plan": [rel], "answer": obj})

    chains_2hop = [
        ("dog", "chase", "cat", "catch", "mouse"),
        ("cat", "catch", "mouse", "eat", "cheese"),
        ("john", "give", "mary", "read", "book"),
    ]
    for e1, r1, e2, r2, e3 in chains_2hop:
        sent = f"the {e1} {r1} the {e2} . the {e2} {r2} the {e3} ."
        for _ in range(15):
            data.append({"sentence": sent, "triplets": [(e1, r1, e2), (e2, r2, e3)], "plan": [r1, r2], "answer": e3})

    for _ in range(num_double):
        e1, e2, e3 = random.sample(entities, 3)
        r1, r2 = random.sample(relations, 2)
        sent = f"the {e1} {r1} the {e2} . the {e2} {r2} the {e3} ."
        data.append({"sentence": sent, "triplets": [(e1, r1, e2), (e2, r2, e3)], "plan": [r1, r2], "answer": e3})

    random.shuffle(data)
    return data

In [64]:
# @title ✅ SEL 6: Agent System (CognitiveAgent, ConsensusEngine, DebateModerator, AutonomousAgent)
# ============================================================================

# ----------------------------------------------------------------------------
# DocumentIngestPipeline: Multi-format knowledge bridge (Audit-Ready)
# ----------------------------------------------------------------------------
class DocumentIngestPipeline:
    def __init__(self, mem: 'AnalyticMemory'):
        self.mem = mem
        self.kw_router: Dict[str, str] = {}

    def register_route(self, keyword: str, module_id: str) -> None:
        self.kw_router[keyword.lower()] = module_id

    def ingest_file(self, filepath: str) -> Dict[str, int]:
        """Returns {'facts_added': int, 'sentences_processed': int, 'errors': int}"""
        stats = {"facts_added": 0, "sentences_processed": 0, "errors": 0}

        try:
            raw_text = DocumentReader.read(filepath)
            doc = nlp(raw_text)
            sentences = [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 5]
        except Exception as e:
            logger.warning(f"Ingest gagal membaca {filepath}: {e}")
            stats["errors"] += 1
            return stats

        if not sentences:
            return stats

        mod_id = self._resolve_module(filepath)
        old_mod = self.mem.active_module_id
        self.mem._switch_module(mod_id)

        processed = 0
        added = 0
        for sent in sentences:
            stats["sentences_processed"] += 1
            try:
                triplet = extract_triplets_strict(sent, self.mem)
                if triplet:
                    subj, rel, obj, conf = triplet
                    if self.mem.add_fact(subj, rel, obj, confidence=conf):
                        stats["facts_added"] += 1
                        processed += 1
                        if conf >= 0.9:
                            self.mem.successful_sentences.append(sent)
            except Exception:
                stats["errors"] += 1

        self.mem._switch_module(old_mod)
        return stats

    def _resolve_module(self, filepath: str) -> str:
        fname = os.path.basename(filepath).lower()
        for kw, mid in self.kw_router.items():
            if kw in fname:
                return mid
        return self.mem.active_module_id


# ----------------------------------------------------------------------------
# CognitiveAgent: Base agent with memory and reasoning capabilities
# ----------------------------------------------------------------------------
class CognitiveAgent:
    def __init__(self, agent_id: str, name: str, knowledge_source: str,
                 dim: int = D, corpus: List[str] = None):
        self.agent_id = agent_id
        self.name = name
        self.knowledge_source = knowledge_source
        self.memory = AnalyticMemory(dim=dim)
        self.model = None
        self.retriever = None
        self.corpus = corpus if corpus else []
        self.tokenizer = None
        self.success_count = 0
        self.total_queries = 0
        self.agent_confidence = 0.8

    def initialize(self, model: 'IntuitionEncoderConditioned', tokenizer: 'DynamicTokenizer') -> None:
        self.model = model
        self.tokenizer = tokenizer
        if self.corpus:
            self.retriever = NeuralRetriever(model, self.corpus, tokenizer)

    def learn_fact(self, subj: str, rel: str, obj: str, confidence: float = 1.0, inferred: bool = False) -> None:
        self.memory.add_fact(subj, rel, obj, inferred=inferred, confidence=confidence)

    def learn_rule(self, r1: str, r2: str, r3: str, confidence: float) -> None:
        self.memory.rule_miner.rules[(r1, r2)] = (r3, confidence)
        self.memory.rules_mined = True

    def answer(self, question: str, plan: List[str], verbose: bool = False) -> Dict[str, Any]:
        result = interactive_qa_with_active_learning(
            question, plan, expected_answer=None,
            mem=self.memory,
            model=self.model,
            retriever=self.retriever,
            tokenizer=self.tokenizer,
            conf_threshold=0.0,
            interactive=False,
            verbose=verbose
        )
        self.total_queries += 1
        self.agent_confidence = max(0.1, self.agent_confidence * 0.9 + 0.1 * result['confidence'])
        return {
             "agent_id": self.agent_id,
             "agent_name": self.name,
             "answer": result["answer"],
             "confidence": result["confidence"],
             "explanation": result["explanation"],
             "trace": result.get("trace", []),
             "agent_confidence": self.agent_confidence
        }


# ----------------------------------------------------------------------------
# ConsensusEngine: Resolves multi-agent answers via weighted voting
# ----------------------------------------------------------------------------
class ConsensusEngine:
    def __init__(self, min_agreement_ratio: float = 0.5, confidence_threshold: float = 0.5):
        self.min_agreement_ratio = min_agreement_ratio
        self.confidence_threshold = confidence_threshold

    def resolve(self, answers: List[Dict]) -> Dict[str, Any]:
        answer_groups = defaultdict(list)
        for ans in answers:
            answer_groups[ans["answer"]].append(ans)

        weighted_conf = {}
        for ans_str, group in answer_groups.items():
            total_conf = sum(a["confidence"] * a.get("agent_confidence", 0.8) for a in group)
            weighted_conf[ans_str] = total_conf

        if not weighted_conf:
            return {
                 "answer": "UNKNOWN", "confidence": 0.0, "consensus": False,
                 "agreement_ratio": 0.0, "supporting_agents": [],
                 "explanation": "Tidak ada agen yang memberikan jawaban.",
                 "all_answers": answers
            }

        best_answer = max(weighted_conf, key=weighted_conf.get)
        best_group = answer_groups[best_answer]
        total_agents = len(answers)
        supporting_agents = len(best_group)
        agreement_ratio = supporting_agents / total_agents
        final_confidence = weighted_conf[best_answer] / total_agents
        consensus_reached = (agreement_ratio >= self.min_agreement_ratio and
                             final_confidence >= self.confidence_threshold)

        combined_explanation = "\n\n".join([
            f"[{a['agent_name']}] (conf={a['confidence']:.2f}): {a['explanation']}"
            for a in best_group
        ])

        return {
             "answer": best_answer,
             "confidence": final_confidence,
             "consensus": consensus_reached,
             "agreement_ratio": agreement_ratio,
             "supporting_agents": [a["agent_id"] for a in best_group],
             "explanation": combined_explanation,
             "all_answers": answers
        }


# ----------------------------------------------------------------------------
# DebateModerator: Coordinates multi-agent debate and consensus
# ----------------------------------------------------------------------------
class DebateModerator:
    def __init__(self, agents: List[CognitiveAgent], consensus_engine: ConsensusEngine):
        self.agents = agents
        self.consensus_engine = consensus_engine

    def query(self, question: str, plan: List[str], verbose: bool = True) -> Dict[str, Any]:
        if verbose:
            print(f"\n🎤 Moderator: Mengajukan '{question}' ke {len(self.agents)} agen... ")
        answers = []
        for agent in self.agents:
            ans = agent.answer(question, plan, verbose=False)
            answers.append(ans)
            if verbose:
                print(f"  🗣️ {agent.name} -> {ans['answer']} (conf={ans['confidence']:.2f}) ")

        resolution = self.consensus_engine.resolve(answers)
        if verbose:
            if resolution["consensus"]:
                print(f"\n✅ Konsensus: {resolution['answer']} (conf={resolution['confidence']:.2f}) ")
                print(f"   Didukung: {', '.join(resolution['supporting_agents'])} ")
            else:
                print(f"\n❌ Tidak ada konsensus. Jawaban terkuat: {resolution['answer']} (conf={resolution['confidence']:.2f}) ")
        return resolution


# ----------------------------------------------------------------------------
# AutonomousAgent: Self-improving agent with background learning loop
# ----------------------------------------------------------------------------
class AutonomousAgent:
    def __init__(self, agent: CognitiveAgent, scheduler_interval: int = 10, watch_folder: Optional[str] = None):
        self.agent = agent
        self.scheduler_interval = scheduler_interval
        self.running = False
        self.thread = None
        self.watch_folder = watch_folder
        self.last_scan_time = 0.0
        self._lock = threading.Lock()
        self.last_total_facts = self._count_total_facts()
        self.ingest_pipeline = DocumentIngestPipeline(agent.memory) if watch_folder else None

    def _count_total_facts(self) -> int:
        total = 0
        for slot in self.agent.memory.slots:
            total += len(slot.direct_facts)
        return total

    def start(self) -> None:
        self.running = True
        self.thread = threading.Thread(target=self._run_loop, daemon=True)
        self.thread.start()
        logger.info("🤖 AutonomousAgent dimulai. Sistem akan belajar secara mandiri. ")

    def stop(self) -> None:
        self.running = False
        if self.thread:
            self.thread.join(timeout=2)
        logger.info("🤖 AutonomousAgent dihentikan. ")

    def _run_loop(self) -> None:
        while self.running:
            try:
                with self._lock:
                    current_total = self._count_total_facts()
                    if current_total > self.last_total_facts:
                        self.agent.memory.mine_rules()
                        self.last_total_facts = current_total
                        print("   ⛏️  Fakta baru terdeteksi, mining aturan... ")

                    self.consolidate_memory()
                    if len(self.agent.memory.error_log) > 0:
                        self.self_assessment()

                    # === NEW: WATCH FOLDER INGESTION ===
                    if self.ingest_pipeline and os.path.isdir(self.watch_folder):
                        curr_time = time.time()
                        files = [f for f in os.listdir(self.watch_folder)
                                 if os.path.isfile(os.path.join(self.watch_folder, f))]

                        new_files = [
                            f for f in files
                            if os.path.getmtime(os.path.join(self.watch_folder, f)) > self.last_scan_time
                        ]

                        if new_files:
                            for fname in new_files:
                                path = os.path.join(self.watch_folder, fname)
                                stats = self.ingest_pipeline.ingest_file(path)
                                print(f"   📥 Ingest '{fname}': +{stats['facts_added']} fakta, {stats['sentences_processed']} kalimat")

                            self.last_scan_time = curr_time
                            print("   ⚠️ Fakta baru terdeteksi, triggering rule mining...")
                            self.agent.memory.mine_rules()
                            self.last_total_facts = self._count_total_facts()

                    # === SELF‑IMPROVING SPACY (INTEGRASI FINAL) ===
                    if 'SelfImprovingParser' in globals():
                        if not hasattr(self, 'spacy_improver'):
                            self.spacy_improver = SelfImprovingParser(self.agent.memory, min_samples=100, cooldown_seconds=600)
                        self.spacy_improver.step()

            except Exception as e:
                logger.warning(f"⚠️ Error dalam autonomous loop: {e} ")
            time.sleep(self.scheduler_interval)

    def consolidate_memory(self) -> None:
        removed = 0
        for slot in self.agent.memory.slots:
            to_remove = []
            for (s, o), conf in list(slot.inferred_confidences.items()):
                if conf < 0.3:
                    to_remove.append((s, o))
            for (s, o) in to_remove:
                slot.inferred_facts.discard((s, o))
                del slot.inferred_confidences[(s, o)]
                if s in slot.inferred_subj_to_objs:
                    slot.inferred_subj_to_objs[s] = [
                        (o2, c2) for (o2, c2) in slot.inferred_subj_to_objs[s] if o2 != o
                    ]
                    if not slot.inferred_subj_to_objs[s]:
                        del slot.inferred_subj_to_objs[s]
                removed += 1
        if removed > 0:
            print(f"   🧹 Konsolidasi memori: menghapus {removed} fakta inferred lemah ")

    def self_assessment(self) -> None:
        with self._lock:
            if not self.agent.memory.error_log:
                return
            recent_errors = [e for e in self.agent.memory.error_log if e.get('confidence', 0) < 0.7]
            if not recent_errors:
                return
            error = random.choice(recent_errors)
            question = error['question']
            expected = error['expected']
            print(f"   🎯 Latihan mandiri: mencoba '{question}' (expected: {expected}) ")

            q_lower = question.lower().replace("?", "").strip()
            tokens = q_lower.split()
            plan = []
            known_relations = self.agent.memory.rel_mgr.relations
            for tok in tokens:
                base = irregular_verbs.get(tok, tok)
                if base in known_relations:
                    plan.append(base)
                elif tok in known_relations:
                    plan.append(tok)
            if not plan and error.get('relation'):
                plan = [error['relation']]
            if not plan:
                plan = ["chase"]

            result = interactive_qa_with_active_learning(
                question=question,
                plan=plan,
                expected_answer=expected,
                mem=self.agent.memory,
                model=self.agent.model,
                retriever=self.agent.retriever,
                tokenizer=self.agent.tokenizer,
                conf_threshold=0.5,
                interactive=False
            )
            if result['correct']:
                self.agent.memory.error_log.remove(error)
                print(f"   ✅ Latihan mandiri berhasil: {question} (plan={plan}) ")
            else:
                print(f"   ❌ Latihan mandiri masih gagal: {result['answer']} (conf={result['confidence']:.2f}, plan={plan}) ")

In [65]:
# @title ✅ SEL 7 FINAL: Open-World Integration (Production‑Ready, Fixed)
# ============================================================================
from urllib.parse import quote
import requests
import re
from typing import List, Optional, Tuple, Dict, Any

# ---------------------------------------------------------------------------
# Utility: Clean Answer (remove noise)
# ---------------------------------------------------------------------------
def clean_answer(text: Optional[str]) -> Optional[str]:
    if not text:
        return text
    text = re.sub(r'\s*(?:sometime|between|in|during|at)\s+[\d\s\w,]*', '', text, flags=re.I)
    text = text.rstrip('.')
    return text.strip() or None

# ---------------------------------------------------------------------------
# SimpleFactExtractor (Fallback Deterministik)
# ---------------------------------------------------------------------------
class SimpleFactExtractor:
    def __init__(self):
        self.patterns = [
            re.compile(r"(?P<subj>[A-Za-zÀ-ÿ\s\.]+?)\s+(?:is|was|are|were)\s+(?:the\s+)?(?P<rel>capital|headquarters|birthplace)\s+of\s+(?P<obj>[A-Za-zÀ-ÿ\s\.]+)", re.IGNORECASE),
            re.compile(r"(?P<obj>[A-Za-zÀ-ÿ\s\.]+?)\s+(?:is|was)\s+located\s+in\s+(?P<subj>[A-Za-zÀ-ÿ\s\.]+)", re.IGNORECASE),
            re.compile(r"(?P<obj>[A-Za-zÀ-ÿ\s\.]+?)\s+was\s+(?:written|authored|composed)\s+by\s+(?P<subj>[A-Za-zÀ-ÿ\s\.]+)", re.IGNORECASE),
            re.compile(r"(?P<subj>[A-Za-zÀ-ÿ\s\.]+?)\s+(?:wrote|authored|composed)\s+(?P<obj>[A-Za-zÀ-ÿ\s\.]+)", re.IGNORECASE),
        ]
        self.rel_map = {
            "capital": "be", "headquarters": "be", "birthplace": "be",
            "written": "write", "authored": "write", "composed": "compose"
        }

    def extract(self, question: str, context: str, mem: 'AnalyticMemory') -> Optional[Dict[str, Any]]:
        if not context or not question:
            return None
        sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', context) if len(s.strip()) > 15]
        q_lower = question.lower()
        for sent in sentences:
            for pattern in self.patterns:
                m = pattern.search(sent)
                if m:
                    subj = normalize_entity_name(m.group('subj'))
                    obj = normalize_entity_name(m.group('obj'))
                    if subj and obj:
                        rel = self.rel_map.get(m.group('rel'), 'be')
                        mem.ent_mgr.get_or_create(subj)
                        mem.ent_mgr.get_or_create(obj)
                        return {
                            'answer': obj,
                            'confidence': 0.85,
                            'triplet': (subj, rel, obj),
                            'method': 'pattern_match',
                            'source_snippet': sent
                        }
        if 'nlp' in globals():
            doc = nlp(context)
            q_words = set(q_lower.replace('?', '').split()) - {'what','is','the','of','who','did','where'}
            best_token, best_score = None, -1
            for sent in doc.sents:
                for token in sent:
                    if token.pos_ in ('NOUN', 'PROPN') and not token.is_stop:
                        score = sum(1 for w in q_words if w in token.sent.text.lower())
                        if token.dep_ in ('attr','dobj','pobj','ROOT'):
                            score += 2
                        if score > best_score:
                            best_score, best_token = score, token
            if best_token and best_score >= 2:
                ans = normalize_entity_name(best_token.text)
                mem.ent_mgr.get_or_create(ans)
                return {'answer': ans, 'confidence': 0.65, 'triplet': None, 'method': 'dep_fallback', 'source_snippet': best_token.sent.text.strip()}
        return None

# ---------------------------------------------------------------------------
# OpenWorldRetriever (dynamic vocabulary sync)
# ---------------------------------------------------------------------------
class OpenWorldRetriever(NeuralRetriever):
    def __init__(self, model: Optional[IntuitionEncoderConditioned],
                 tokenizer: Optional[DynamicTokenizer],
                 initial_corpus: Optional[List[str]] = None,
                 max_corpus_size: int = 500):
        super().__init__(model, initial_corpus or [], tokenizer)
        self.max_corpus_size = max_corpus_size

    def add_sentences(self, new_sentences: List[str]) -> None:
        if not new_sentences:
            return
        existing = set(self.corpus)
        unique_new = [s for s in new_sentences if s not in existing]
        if not unique_new:
            return
        for sent in unique_new:
            for tok in sent.lower().replace('.', ' .').replace('?', ' ?').split():
                if tok not in self.tokenizer.word2idx:
                    self.tokenizer._on_new_token(tok)
        total_after = len(self.corpus) + len(unique_new)
        if total_after > self.max_corpus_size:
            remove_count = total_after - self.max_corpus_size
            self.corpus = self.corpus[remove_count:]
            if self.corpus_embeddings is not None:
                self.corpus_embeddings = self.corpus_embeddings[remove_count:]
        if self.model is not None and self.tokenizer is not None:
            self.model.eval()
            new_embs = []
            with torch.no_grad():
                for sent in unique_new:
                    ids = torch.tensor(self.tokenizer.tokenize(sent), device=device).unsqueeze(0)
                    mask = torch.ones_like(ids).float()
                    out = self.model(ids, mask)
                    new_embs.append(F.normalize(out["pooled"], p=2, dim=-1))
            new_emb_tensor = torch.cat(new_embs, dim=0)
            self.corpus.extend(unique_new)
            if self.corpus_embeddings is None:
                self.corpus_embeddings = new_emb_tensor
            else:
                self.corpus_embeddings = torch.cat([self.corpus_embeddings, new_emb_tensor], dim=0)
        else:
            self.corpus.extend(unique_new)
        logger.info(f"📚 OpenWorld: +{len(unique_new)} kalimat, total: {len(self.corpus)}")

    def fetch_external_knowledge(self, query_text: Optional[str] = None, top_k: int = 2) -> List[str]:
        lines = safe_read_lines("external_knowledge.txt")
        return lines[:min(top_k, len(lines))]

    def extend_from_file(self, filepath: str) -> None:
        sents = safe_read_lines(filepath)
        if sents:
            self.add_sentences(sents)

# ---------------------------------------------------------------------------
# WikipediaSource (smart query untuk ibukota)
# ---------------------------------------------------------------------------
class WikipediaSource:
    def __init__(self, lang: str = "en"):
        self.api_url = f"https://{lang}.wikipedia.org/w/api.php"
        self.headers = {'User-Agent': 'BrainNN/1.0'}

    def fetch_summary(self, keyword: str, max_sentences: int = 3) -> Optional[str]:
        try:
            safe_kw = quote(keyword)
            r = requests.get(f"{self.api_url}?action=query&list=search&srsearch={safe_kw}&srlimit=1&format=json",
                             headers=self.headers, timeout=10)
            if r.status_code != 200: return None
            sr = r.json().get('query', {}).get('search', [])
            if not sr: return None
            title = sr[0]['title']
            r2 = requests.get(f"{self.api_url}?action=query&prop=extracts&exintro&explaintext&titles={quote(title)}&format=json",
                              headers=self.headers, timeout=10)
            pages = r2.json().get('query', {}).get('pages', {})
            page = next((p for p in pages.values() if p.get('pageid')), None)
            if not page: return None
            extract = page.get('extract', '')
            sentences = [s.strip() for s in extract.split('. ') if s.strip()]
            return '. '.join(sentences[:max_sentences]) + '.'
        except Exception as e:
            logger.warning(f"Wikipedia fetch error for '{keyword}': {e}")
            return None

# ---------------------------------------------------------------------------
# OpenWorldRetrieverWithWiki (smart query)
# ---------------------------------------------------------------------------
class OpenWorldRetrieverWithWiki(OpenWorldRetriever):
    def __init__(self, *args, wiki_lang: str = "en", **kwargs):
        super().__init__(*args, **kwargs)
        self.wiki = WikipediaSource(lang=wiki_lang)

    def fetch_external_knowledge(self, query_text: Optional[str] = None, top_k: int = 2) -> List[str]:
        results = super().fetch_external_knowledge(query_text, top_k)
        if query_text:
            keywords = self._extract_keywords(query_text)
            for kw in keywords:
                actual_kw = kw
                if "capital" in query_text.lower() and kw != "capital":
                    actual_kw = f"Capital of {kw}"
                wiki_text = self.wiki.fetch_summary(actual_kw)
                if wiki_text:
                    results.append(wiki_text + " .")
        seen = set()
        uniq = []
        for r in results:
            key = r.lower().strip()
            if key not in seen:
                seen.add(key)
                uniq.append(r)
        return uniq[:top_k*2]

    def _extract_keywords(self, question: str) -> List[str]:
        if 'nlp' in globals():
            doc = nlp(question)
            words = [t.text.lower() for t in doc if t.pos_ in ('NOUN', 'PROPN') and not t.is_stop]
        else:
            words = [w for w in question.lower().replace('?','').split() if len(w) > 2]
        seen = set()
        return [w for w in words if not (w in seen or seen.add(w))][:3]

# ---------------------------------------------------------------------------
# OpenWorldCognitiveAgent (base class)
# ---------------------------------------------------------------------------
class OpenWorldCognitiveAgent(CognitiveAgent):
    def __init__(self, agent_id: str, name: str, knowledge_source: str,
                 dim: int = D, corpus: Optional[List[str]] = None,
                 max_corpus_size: int = 500):
        super().__init__(agent_id, name, knowledge_source, dim=dim, corpus=None)
        self.open_retriever = OpenWorldRetriever(None, None, initial_corpus=corpus, max_corpus_size=max_corpus_size)

    def initialize(self, model: IntuitionEncoderConditioned, tokenizer: DynamicTokenizer) -> None:
        self.model = model
        self.tokenizer = tokenizer
        self.open_retriever.model = model
        self.open_retriever.tokenizer = tokenizer
        if self.open_retriever.corpus and self.open_retriever.corpus_embeddings is None:
            self.open_retriever._encode_corpus()

    def learn_from_corpus(self) -> None:
        if not self.open_retriever.corpus:
            print("   ℹ️ Korpus kosong.")
            return
        for sent in self.open_retriever.corpus:
            triplets = extract_triplets_from_text(self.model, sent, self.memory)
            for subj, rel, obj, conf in triplets:
                if self.memory.add_fact(subj, rel, obj, inferred=False, confidence=conf):
                    print(f"   📘 Belajar dari korpus: {subj} --[{rel}]--> {obj} (conf={conf:.2f})")

    def answer(self, question: str, plan: List[str], verbose: bool = False) -> Dict[str, Any]:
        result = interactive_qa_with_active_learning(
            question, plan, expected_answer=None,
            mem=self.memory,
            model=self.model,
            retriever=self.open_retriever,
            tokenizer=self.tokenizer,
            conf_threshold=0.0,
            interactive=False,
            verbose=verbose
        )
        if result['confidence'] < 0.4:
            external_sents = self.open_retriever.fetch_external_knowledge(question)
            if external_sents:
                self.open_retriever.add_sentences(external_sents)
                for sent in external_sents:
                    triplets = extract_triplets_from_text(self.model, sent, self.memory)
                    for subj, rel, obj, conf in triplets:
                        self.memory.add_fact(subj, rel, obj, inferred=False, confidence=conf)
                result = interactive_qa_with_active_learning(
                    question, plan, expected_answer=None,
                    mem=self.memory,
                    model=self.model,
                    retriever=self.open_retriever,
                    tokenizer=self.tokenizer,
                    conf_threshold=0.0,
                    interactive=False,
                    verbose=False
                )
        self.total_queries += 1
        self.agent_confidence = max(0.1, self.agent_confidence * 0.9 + 0.1 * result['confidence'])
        return {
            "agent_id": self.agent_id,
            "agent_name": self.name,
            "answer": result["answer"],
            "confidence": result["confidence"],
            "explanation": result["explanation"],
            "trace": result.get("trace", []),
            "agent_confidence": self.agent_confidence
        }

# ---------------------------------------------------------------------------
# OpenWorldCognitiveAgentWithSharedVocab (tambahan shared ent_mgr/rel_mgr + fallback)
# ---------------------------------------------------------------------------
class OpenWorldCognitiveAgentWithSharedVocab(OpenWorldCognitiveAgent):
    def __init__(self, agent_id: str, name: str, knowledge_source: str,
                 dim: int = D, corpus: Optional[List[str]] = None,
                 max_corpus_size: int = 500,
                 ent_mgr: Optional[EntityManager] = None,
                 rel_mgr: Optional[RelationManager] = None,
                 fallback_extractor: Optional[SimpleFactExtractor] = None):
        super().__init__(agent_id, name, knowledge_source, dim=dim, corpus=None,
                         max_corpus_size=max_corpus_size)
        if ent_mgr: self.memory.ent_mgr = ent_mgr
        if rel_mgr: self.memory.rel_mgr = rel_mgr
        self.fallback_extractor = fallback_extractor or SimpleFactExtractor()

    def initialize(self, model: IntuitionEncoderConditioned, tokenizer: DynamicTokenizer) -> None:
        super().initialize(model, tokenizer)
        model.ent_mgr = self.memory.ent_mgr
        model.rel_mgr = self.memory.rel_mgr

    def answer(self, question: str, plan: List[str], verbose: bool = False) -> Dict[str, Any]:
        result = super().answer(question, plan, verbose)
        if result['confidence'] < 0.4 and self.fallback_extractor:
            ext_sents = self.open_retriever.fetch_external_knowledge(question)
            if ext_sents:
                context = " . ".join(ext_sents)
                fallback = self.fallback_extractor.extract(question, context, self.memory)
                if fallback and fallback['confidence'] > result['confidence']:
                    result['answer'] = fallback['answer']
                    result['confidence'] = fallback['confidence']
                    result['explanation'] = (
                        f"📌 Fallback ({fallback['method']}): {fallback['answer']} (conf={fallback['confidence']:.2f})"
                    )
                    if fallback.get('triplet'):
                        s, r, o = fallback['triplet']
                        self.memory.add_fact(s, r, o, inferred=False, confidence=fallback['confidence'])
        if result.get('answer'):
            cleaned = clean_answer(result['answer'])
            if cleaned:
                result['answer'] = cleaned
            else:
                result['answer'] = "UNKNOWN"
                result['confidence'] = 0.0
        self.total_queries += 1
        self.agent_confidence = max(0.1, self.agent_confidence * 0.9 + 0.1 * result['confidence'])
        return result

# ---------------------------------------------------------------------------
# Helper & Demo
# ---------------------------------------------------------------------------
def ensure_external_knowledge_file(filepath: str = "external_knowledge.txt") -> bool:
    if os.path.exists(filepath):
        return True
    default = ["the wolf caught the rabbit .", "the rabbit ate the carrot .", "the cat chased the mouse .", "the dog chased the ball ."]
    with open(filepath, 'w') as f:
        f.write("\n".join(default) + "\n")
    return False

def demo_open_world_final(model: IntuitionEncoderConditioned,
                          tokenizer: DynamicTokenizer,
                          ent_mgr: EntityManager,
                          rel_mgr: RelationManager) -> None:
    print("\n" + "="*70)
    print("TAHAP 10 – OPEN‑WORLD KNOWLEDGE INTEGRATION (FINAL)")
    print("="*70)
    ensure_external_knowledge_file()
    agent = OpenWorldCognitiveAgentWithSharedVocab(
        "A4", "🌐 Open Agent", "Dynamic",
        corpus=["the dog chased the cat .", "the cat caught the mouse ."],
        ent_mgr=ent_mgr, rel_mgr=rel_mgr
    )
    agent.initialize(model, tokenizer)
    agent.learn_fact("dog", "chase", "cat")
    agent.learn_fact("cat", "catch", "mouse")
    agent.open_retriever.extend_from_file("external_knowledge.txt")
    agent.learn_from_corpus()
    questions = [
        ("what did the wolf catch ?", ["catch"]),
        ("what did the rabbit eat ?", ["eat"]),
        ("what did the dog chase ?", ["chase"]),
    ]
    print("\n🔍 Uji Open-World Agent:")
    for q, plan in questions:
        ans = agent.answer(q, plan)
        print(f"  ❓ {q}")
        print(f"  ➜ {ans['answer']} (conf={ans['confidence']:.2f})")
        if ans['explanation']: print(f"  {ans['explanation']}")
        print()
    print("🤖 Autonomous loop (12 detik)...")
    auto = AutonomousAgent(agent, scheduler_interval=5)
    auto.start()
    time.sleep(12)
    auto.stop()
    print("\n✅ TAHAP 10 SELESAI.")

def demo_wikipedia_integration(model: IntuitionEncoderConditioned,
                               tokenizer: DynamicTokenizer,
                               ent_mgr: EntityManager,
                               rel_mgr: RelationManager,
                               test_online: bool = False) -> None:
    print("\n" + "="*70)
    print("TAHAP 11 – WIKIPEDIA LIVE (OPTIONAL)")
    print("="*70)
    wiki_retriever = OpenWorldRetrieverWithWiki(model=model, tokenizer=tokenizer,
                                                initial_corpus=["the dog chased the cat ."])
    agent = OpenWorldCognitiveAgentWithSharedVocab(
        "A5", "🌍 Wikipedia Agent", "Live Wikipedia",
        ent_mgr=ent_mgr, rel_mgr=rel_mgr
    )
    agent.initialize(model, tokenizer)
    agent.open_retriever = wiki_retriever
    agent.learn_fact("dog", "chase", "cat")
    questions = [("what did the dog chase ?", ["chase"])]
    if test_online:
        questions += [
            ("what is the capital of france ?", ["be"]),
            ("who wrote hamlet ?", ["write"]),
        ]
    print("\n🔍 Uji:")
    for q, plan in questions:
        ans = agent.answer(q, plan)
        print(f"  ❓ {q}")
        print(f"  ➜ {ans['answer']} (conf={ans['confidence']:.2f})")
        if ans['explanation']: print(f"  {ans['explanation']}")
        print()
    print("✅ TAHAP 11 SELESAI.")

In [66]:
# @title ✅ SEL 8: Demo Functions & Main Execution (Final)
# ============================================================================
import time

# ----------------------------------------------------------------------------
# Tahap 7.5: Explainable Decisions & Active Learning Demo
# ----------------------------------------------------------------------------
def run_tahap7_5():
    print("\n" + "="*70)
    print("TAHAP 7.5 – EXPLAINABLE DECISIONS & ACTIVE LEARNING (FULLY OTONOM)")
    print("="*70)

    ent_mgr = EntityManager(D)
    rel_mgr = RelationManager()
    tokenizer = DynamicTokenizer(ent_mgr, rel_mgr)

    data = generate_multi_hop_dataset(ent_mgr, rel_mgr, num_single=200, num_double=100, num_triple=50)
    dataset = ReasoningDataset(data, tokenizer, ent_mgr, rel_mgr)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    model = IntuitionEncoderConditioned(tokenizer, D, ent_mgr, rel_mgr).to(device)
    optimizer = optim.Adam(model.parameters(), lr=NEURAL_LR)

    print("\n🧠 Melatih Encoder...")
    train_extractor(model, dataloader, optimizer, EPOCHS)

    CORPUS = [
        "the dog chased the cat .",
        "the cat caught the mouse .",
        "the mouse ate the cheese .",
        "the cheese was in the kitchen .",
        "john gave the book to mary .",
        "mary read the book in the park .",
        "alice bought flowers in the shop .",
        "the child saw the ball in the room .",
        "the mother gave the book to the child .",
        "the father read the book to the child .",
    ]
    retriever = NeuralRetriever(model, CORPUS, tokenizer)

    print("\n📚 Mempersiapkan data train (mining) dan test (evaluasi)...")
    base_facts = [
        ("dog", "chase", "cat"),
        ("cat", "catch", "mouse"),
        ("cat", "chase", "mouse"),
        ("mouse", "eat", "cheese"),
        ("cheese", "be", "kitchen"),
        ("mother", "give", "child"),
        ("father", "read", "child"),
    ]

    mem_train = AnalyticMemory(dim=D)
    for subj, rel, obj in base_facts:
        mem_train.add_fact(subj, rel, obj)
    mem_train.add_fact("dog", "eat", "mouse")

    mem_test = AnalyticMemory(dim=D)
    for subj, rel, obj in base_facts:
        mem_test.add_fact(subj, rel, obj)

    print("\n⛏️  Mining aturan heterogen dari data train...")
    mem_train.mine_rules()
    rules_found = mem_train.rule_miner.rules
    if rules_found:
        for (r1, r2), (r3, conf) in rules_found.items():
            print(f"    Aturan ditemukan: ({r1}, {r2}) → {r3} (conf={conf:.2f})")
    else:
        print("    Tidak ada aturan yang memenuhi threshold.")

    mem_test.rule_miner.rules = rules_found.copy()
    mem_test.rules_mined = True

    print("\n🧠 Menerapkan aturan pada memori test...")
    added_homo = mem_test.apply_ilp()
    added_hetero = mem_test.apply_hetero_ilp()
    print(f"    Fakta inferred: +{added_homo} transitif, +{added_hetero} heterogen")

    print("\n" + "="*70)
    print("🔍 DEMO EXPLAINABLE QA (TANPA INTERAKSI)")
    print("="*70)

    question = "what did the dog eat ?"
    plan = ["eat"]
    expected = "mouse"
    result = interactive_qa_with_active_learning(
        question, plan, expected, mem_test, model, retriever, tokenizer,
        conf_threshold=GLOBAL_CONFIDENCE_THRESHOLD, interactive=False
    )
    print(f"\nQ: {question}")
    print(result["explanation"])
    print(f"Status: {'✅' if result['correct'] else '❌'} (confidence={result['confidence']:.2f})")

    print("\n" + "="*70)
    print("🔍 DEMO ACTIVE LEARNING (CONFIDENCE RENDAH)")
    print("="*70)
    mem_low = AnalyticMemory(dim=D)
    mem_low.add_fact("dog", "chase", "cat", inferred=True, confidence=0.3)
    print("Situasi: Memori hanya punya 'dog chase cat' dengan confidence 0.3")
    question2 = "what did the dog chase ?"
    plan2 = ["chase"]
    expected2 = "cat"
    result2 = interactive_qa_with_active_learning(
        question2, plan2, expected2, mem_low, model, retriever, tokenizer,
        conf_threshold=0.5, interactive=False
    )
    print(f"\nHasil akhir: {result2['answer']} (conf={result2['confidence']:.2f})")

    print("\n" + "="*70)
    print("📊 ANALISIS PERFORMA RELASI")
    print("="*70)
    perf_analysis = mem_test.analyze_relation_performance()
    for rel, stats in perf_analysis.items():
        print(f"  {rel}: akurasi={stats['accuracy']:.2f}, total={stats['total_attempts']}")

    print("\n" + "="*70)
    print("⚙️ PARAMETER DINAMIS SETELAH ADAPTASI")
    print("="*70)
    params = mem_test.meta_controller.params
    for param_name, value in params.items():
        print(f"  {param_name}: {value}")

    print("\n" + "="*70)
    print("✅ TAHAP 7.5 SELESAI.")
    print("="*70)

    return model, tokenizer, ent_mgr, rel_mgr

# ============================================================================
# MAIN EXECUTION
# ============================================================================
if __name__ == "__main__":
    model, tokenizer, ent_mgr, rel_mgr = run_tahap7_5()

    # ---- TAHAP 8: Multi-Agent Debate ----
    print("\n" + "="*70)
    print("TAHAP 8: MULTI-AGENT DEBATE")
    print("="*70)
    corpus_clean = ["the dog chased the cat .", "the cat chased the mouse .", "the mouse ate the cheese ."]
    corpus_noisy = ["the dog chased the cat .", "the cat chased the mouse .", "the dog chased the ball ."]
    agent_clean = CognitiveAgent("A1", "🧹 Clean Agent", "High-quality data", corpus=corpus_clean)
    agent_noisy = CognitiveAgent("A2", "📢 Noisy Agent", "Web crawl (some errors)", corpus=corpus_noisy)
    agent_reasoner = CognitiveAgent("A3", "🧠 Reasoner Agent", "Logical deduction only", corpus=[])
    agents = [agent_clean, agent_noisy, agent_reasoner]
    for a in agents: a.initialize(model, tokenizer)
    for a in agents:
        a.learn_fact("dog", "chase", "cat")
        a.learn_fact("cat", "chase", "mouse")
    agent_reasoner.learn_rule("chase", "chase", "chase", 1.0)
    consensus = ConsensusEngine(min_agreement_ratio=0.5, confidence_threshold=0.4)
    moderator = DebateModerator(agents, consensus)
    print(moderator.query("what did the dog chase after the cat ?", ["chase", "chase"])["explanation"])

    # ---- TAHAP 9: Autonomous Agent ----
    print("\n" + "="*70)
    print("TAHAP 9: AUTONOMOUS LEARNING LOOP")
    print("="*70)
    demo_agent = agent_clean
    demo_agent.memory.error_log.append({'question':'what did the dog chase ?','expected':'cat','confidence':0.4,'actual':'ball','relation':'chase'})
    auto = AutonomousAgent(demo_agent, scheduler_interval=5)
    auto.start()
    print("🧪 Autonomous loop 15 detik...")
    time.sleep(15)
    auto.stop()
    print("Hasil slot chase:", len(demo_agent.memory.slots[0].direct_facts), "direct,", len(demo_agent.memory.slots[0].inferred_facts), "inferred")

    # ---- TAHAP 10: Open-World ----
    demo_open_world_final(model, tokenizer, ent_mgr, rel_mgr)

    # ---- TAHAP 11 (opsional, offline) ----
    # demo_wikipedia_integration(model, tokenizer, ent_mgr, rel_mgr, test_online=False)


TAHAP 7.5 – EXPLAINABLE DECISIONS & ACTIVE LEARNING (FULLY OTONOM)

🧠 Melatih Encoder...

📚 Mempersiapkan data train (mining) dan test (evaluasi)...

⛏️  Mining aturan heterogen dari data train...
    Aturan ditemukan: (chase, chase) → eat (conf=1.00)
    Aturan ditemukan: (chase, catch) → eat (conf=1.00)

🧠 Menerapkan aturan pada memori test...
    Fakta inferred: +0 transitif, +0 heterogen

🔍 DEMO EXPLAINABLE QA (TANPA INTERAKSI)
    🔄 Retrieval attempt 1...
       Added fact: john --[give]--> book (conf=1.00)
       Added fact: alice --[buy]--> flowers (conf=1.00)
    🔄 Retrieval attempt 2...
       Added fact: john --[give]--> book (conf=1.00)
       Added fact: alice --[buy]--> flowers (conf=1.00)

Q: what did the dog eat ?
Tidak dapat menemukan jawaban.
Status: ❌ (confidence=0.00)

🔍 DEMO ACTIVE LEARNING (CONFIDENCE RENDAH)
Situasi: Memori hanya punya 'dog chase cat' dengan confidence 0.3

Hasil akhir: cat (conf=0.30)

📊 ANALISIS PERFORMA RELASI
  chase: akurasi=1.00, total=2
  

In [67]:
# @title ✅ SEL 9: DocumentReader & DocumentIngestPipeline (Final, Multi-Format & Watch Folder Ready)
# ============================================================================
import os
import sys
import subprocess
from typing import List, Optional, Tuple, Dict, Any

# ---------------------------------------------------------------------------
# Auto‑install pustaka jika belum ada
# ---------------------------------------------------------------------------
def install_if_missing(pkg, import_name=None):
    if import_name is None:
        import_name = pkg.replace("-", "_")
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install_if_missing("pdfplumber", "pdfplumber")
install_if_missing("python-docx", "docx")
install_if_missing("beautifulsoup4", "bs4")

import pdfplumber
from docx import Document
from bs4 import BeautifulSoup

class DocumentReader:
    """
    Pembaca dokumen multi‑format (TXT, PDF, DOCX, HTML).
    """
    @staticmethod
    def read(filepath: str) -> str:
        if not os.path.exists(filepath):
            raise FileNotFoundError(f"File tidak ditemukan: {filepath}")

        ext = os.path.splitext(filepath)[1].lower()
        if ext == ".txt":
            return DocumentReader._read_txt(filepath)
        elif ext == ".pdf":
            return DocumentReader._read_pdf(filepath)
        elif ext == ".docx":
            return DocumentReader._read_docx(filepath)
        elif ext in (".html", ".htm"):
            return DocumentReader._read_html(filepath)
        else:
            raise ValueError(f"Format file tidak didukung: {ext}")

    @staticmethod
    def _read_txt(filepath: str) -> str:
        with open(filepath, "r", encoding="utf-8") as f:
            return f.read()

    @staticmethod
    def _read_pdf(filepath: str) -> str:
        text = []
        with pdfplumber.open(filepath) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text.append(page_text)
        return "\n".join(text)

    @staticmethod
    def _read_docx(filepath: str) -> str:
        doc = Document(filepath)
        paragraphs = [para.text for para in doc.paragraphs if para.text.strip()]
        return "\n".join(paragraphs)

    @staticmethod
    def _read_html(filepath: str) -> str:
        with open(filepath, "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f.read(), "html.parser")
        for tag in soup(["script", "style"]):
            tag.decompose()
        return soup.get_text()


# ---------------------------------------------------------------------------
# DocumentIngestPipeline: Jembatan DocumentReader → KnowledgeModule
# ---------------------------------------------------------------------------
class DocumentIngestPipeline:
    def __init__(self, mem: 'AnalyticMemory'):
        self.mem = mem
        self.kw_router: Dict[str, str] = {}  # kata_kunci -> module_id

    def register_route(self, keyword: str, module_id: str) -> None:
        """Daftarkan kata kunci yang jika muncul di nama file akan mengarahkan ke modul tertentu."""
        self.kw_router[keyword.lower()] = module_id

    def ingest_file(self, filepath: str) -> Dict[str, int]:
        """
        Membaca file, mengekstrak triplet, dan memasukkannya ke KnowledgeModule.
        Mengembalikan statistik: facts_added, sentences_processed, errors.
        """
        stats = {"facts_added": 0, "sentences_processed": 0, "errors": 0}
        try:
            raw_text = DocumentReader.read(filepath)
            doc = nlp(raw_text)
            sentences = [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 5]
        except Exception as e:
            logger.warning(f"Ingest gagal membaca {filepath}: {e}")
            stats["errors"] += 1
            return stats

        if not sentences:
            return stats

        # Tentukan modul tujuan berdasarkan nama file
        mod_id = self._resolve_module(filepath)
        old_mod = self.mem.active_module_id
        self.mem._switch_module(mod_id)

        for sent in sentences:
            stats["sentences_processed"] += 1
            try:
                triplet = extract_triplets_strict(sent, self.mem)
                if triplet:
                    subj, rel, obj, conf = triplet
                    if self.mem.add_fact(subj, rel, obj, confidence=conf):
                        stats["facts_added"] += 1
                        if conf >= 0.9:
                            self.mem.successful_sentences.append(sent)
            except Exception:
                stats["errors"] += 1

        # Kembalikan ke modul semula
        self.mem._switch_module(old_mod)
        return stats

    def _resolve_module(self, filepath: str) -> str:
        fname = os.path.basename(filepath).lower()
        for kw, mid in self.kw_router.items():
            if kw in fname:
                return mid
        return self.mem.active_module_id

In [68]:
# @title ✅ SEL 10: Self‑Improving spaCy Pipeline (Final, Pipeline Format Fixed)
import os, sys, time, threading, subprocess, textwrap, re
import spacy
from spacy.tokens import DocBin

def ensure_base_model(base_model: str = "en_core_web_sm") -> None:
    try:
        spacy.load(base_model)
    except OSError:
        print(f"📥 Mengunduh model dasar: {base_model}")
        subprocess.check_call([sys.executable, "-m", "spacy", "download", base_model])

def is_valid_sentence(sent: str) -> bool:
    sent = sent.strip()
    if not sent:
        return False
    if not re.search(r'[.!?]$', sent):
        return False
    doc = nlp(sent)
    tokens = [t for t in doc if not t.is_space]
    if len(tokens) < 3:
        return False
    if not any(t.pos_ == "VERB" for t in tokens):
        return False
    return True

def clean_sentence(sent: str) -> str:
    sent = sent.strip()
    sent = re.sub(r'\s+', ' ', sent)
    sent = re.sub(r'\s([.!?])', r'\1', sent)
    if not sent.endswith('.'):
        sent = sent.rstrip('.') + '.'
    return sent

def build_training_data(
    sentences: list,
    output_file: str,
    nlp_pipeline: spacy.language.Language,
) -> bool:
    valid_sents = []
    for raw in sentences:
        cleaned = clean_sentence(raw)
        if is_valid_sentence(cleaned):
            valid_sents.append(cleaned)
    if not valid_sents:
        print("⚠️ Tidak ada kalimat valid setelah validasi ketat.")
        return False
    print(f"✅ {len(valid_sents)} kalimat valid (dari {len(sentences)} total).")
    db = DocBin()
    for sent in valid_sents:
        doc = nlp_pipeline(sent)
        db.add(doc)
    os.makedirs(os.path.dirname(output_file) if os.path.dirname(output_file) else ".", exist_ok=True)
    db.to_disk(output_file)
    print(f"📊 Data pelatihan disimpan: {output_file}")
    return True

def fine_tune_spacy_on_data(
    training_file: str,
    output_dir: str = "custom_spacy_model",
    base_model: str = "en_core_web_sm",
    epochs: int = 10,
    dropout: float = 0.1,
    num_samples: int = 0,
    max_steps: int = 500,
) -> bool:
    ensure_base_model(base_model)
    os.makedirs(output_dir, exist_ok=True)

    # Gunakan pipeline tanpa parser jika data terlalu sedikit
    use_parser = num_samples >= 20
    pipeline_str = "tok2vec,tagger,parser" if use_parser else "tok2vec,tagger"
    print(f"ℹ️ Pipeline yang digunakan: {pipeline_str} (karena data={num_samples})")

    config_content = textwrap.dedent(f"""\
    [paths]
    train = "{training_file}"
    dev = "{training_file}"

    [system]

    [nlp]
    lang = "en"
    pipeline = ["tok2vec","tagger"{",\"parser\"" if use_parser else ""}]

    [components]
    [components.tok2vec]
    factory = "tok2vec"

    [components.tagger]
    factory = "tagger"
    """)

    if use_parser:
        config_content += textwrap.dedent("""
    [components.parser]
    factory = "parser"
    """)

    config_content += textwrap.dedent(f"""
    [corpora]
    [corpora.train]
    @readers = "spacy.Corpus.v1"
    path = ${{paths.train}}
    [corpora.dev]
    @readers = "spacy.Corpus.v1"
    path = ${{paths.dev}}

    [training]
    seed = 0
    train_corpus = "corpora.train"
    dev_corpus = "corpora.dev"
    dropout = {dropout}
    accumulate_gradient = 1
    patience = 100
    max_epochs = {epochs}
    max_steps = {max_steps}
    gpu_allocator = "null"
    """)

    config_path = os.path.join(output_dir, "config.cfg")
    with open(config_path, "w") as f:
        f.write(config_content)

    print(f"⏳ Memulai fine‑tuning spaCy ({num_samples} sampel)...")
    cmd = [
        sys.executable, "-m", "spacy", "train",
        config_path,
        "--output", output_dir,
        "--paths.train", training_file,
        "--paths.dev", training_file,
    ]
    try:
        subprocess.run(cmd, check=True, capture_output=True, text=True)
        print("✅ Fine‑tuning selesai.")
        test_nlp = spacy.load(os.path.join(output_dir, "model-best"))
        print("✅ Model-best berhasil dimuat.")
        return True
    except subprocess.CalledProcessError as e:
        print("❌ Fine‑tuning gagal.")
        print("📋 STDOUT:\n", e.stdout[-1500:])
        print("📋 STDERR:\n", e.stderr[-1500:])
        return False

class SelfImprovingParser:
    def __init__(
        self,
        memory: 'AnalyticMemory',
        min_samples: int = 20,
        output_dir: str = "custom_spacy_model",
        base_model: str = "en_core_web_sm",
        cooldown_seconds: float = 300.0,
    ):
        self.memory = memory
        self.min_samples = min_samples
        self.output_dir = output_dir
        self.base_model = base_model
        self.cooldown_seconds = cooldown_seconds

        self.last_improved: float = 0.0
        self._lock = threading.Lock()

        ensure_base_model(base_model)
        self.nlp: spacy.language.Language = spacy.load(base_model)

    def step(self) -> bool:
        sentences = list(set(self.memory.successful_sentences))
        if len(sentences) < self.min_samples:
            print(f"ℹ️ Self‑Improve: data belum cukup ({len(sentences)}/{self.min_samples}).")
            return False

        elapsed = time.time() - self.last_improved
        if self.last_improved > 0 and elapsed < self.cooldown_seconds:
            print(f"⏱️ Cooldown aktif. Training berikutnya dalam {int(self.cooldown_seconds - elapsed)}s.")
            return False

        with self._lock:
            stamp = int(time.time())
            training_file = f"spacy_train_{stamp}.spacy"

            try:
                if not build_training_data(sentences, training_file, self.nlp):
                    return False

                n_samples = len(sentences)
                success = fine_tune_spacy_on_data(
                    training_file=training_file,
                    output_dir=self.output_dir,
                    base_model=self.base_model,
                    num_samples=n_samples,
                    epochs=8,
                    max_steps=500,
                )
                if success:
                    self.nlp = spacy.load(os.path.join(self.output_dir, "model-best"))
                    self.last_improved = time.time()
                    print(f"🔄 Model internal diperbarui (versi: {self.last_improved:.0f})")
                    return True
                return False
            finally:
                if os.path.exists(training_file):
                    os.remove(training_file)
                    print(f"🗑️ File sementara dihapus: {training_file}")

    def reset(self, confirm: bool = False) -> None:
        if not confirm:
            print("⚠️ reset() membutuhkan confirm=True.")
            return
        with self._lock:
            cnt = len(self.memory.successful_sentences)
            self.memory.successful_sentences = []
            print(f"🗑️ Reset: {cnt} kalimat dihapus dari memori.")

# ---------------------------------------------------------------------------
# 5. Demo
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    print("🧪 Uji Self‑Improving spaCy Pipeline – Strict Validation")
    print("=" * 60)

    if 'agent_clean' in globals():
        memory = agent_clean.memory
    else:
        class DummyMemory:
            def __init__(self):
                self.successful_sentences = []
        memory = DummyMemory()

    sample = [
        "The wolf caught the rabbit.",
        "The rabbit ate the carrot.",
        "The dog chased the cat.",
        "The cat caught the mouse.",
        "The mouse ate the cheese.",
        "John gave the book to Mary.",
        "The teacher wrote the answer on the board.",
        "She opened the door carefully.",
        "He read the letter twice.",
        "They finished the project on time.",
    ]
    for s in sample:
        memory.successful_sentences.append(s)

    parser = SelfImprovingParser(memory=memory, min_samples=5, cooldown_seconds=0)
    success = parser.step()
    if success:
        print("🎉 Self‑improvement berhasil!")
    else:
        print("ℹ️ Tidak ada peningkatan kali ini.")

🧪 Uji Self‑Improving spaCy Pipeline – Strict Validation
✅ 10 kalimat valid (dari 10 total).
📊 Data pelatihan disimpan: spacy_train_1777025729.spacy
ℹ️ Pipeline yang digunakan: tok2vec,tagger (karena data=10)
⏳ Memulai fine‑tuning spaCy (10 sampel)...
✅ Fine‑tuning selesai.
✅ Model-best berhasil dimuat.
🔄 Model internal diperbarui (versi: 1777025737)
🗑️ File sementara dihapus: spacy_train_1777025729.spacy
🎉 Self‑improvement berhasil!
